# 09 · Vincular la segmentación de exportadores con ORBIS y FDI

> Por **razón social + país** · motor **rues-linker 0.22.4** · Google Colab

**Qué hace.** Enlaza cada empresa de `Piloto_segmentacion_Empresas_exportadoras.xlsx`
(**Empresa** + **País de Empresa**) con dos hojas de `Base_Piloto3_Descargas_Agroalimentos.xlsx`:
**Orbis** (**Company name Latin alphabet** + **Country**) y **FDI** (**Company name** +
**País de consulta**). Además deduplica cada una de las tres bases por dentro.

**Qué entrega.** Un Excel con el match verificado:

| hoja | contenido |
|---|---|
| `LEEME` · `RESUMEN` | cómo leer el resultado y sus cifras principales |
| `SEG_CONSOLIDADO` | una fila por empresa de la segmentación: métricas sumadas, su mejor enlace en ORBIS y en FDI, y el estado |
| `MATCH_SEG_ORBIS` · `MATCH_SEG_FDI` | todos los enlaces, con clase, origen, puntaje y el contexto del registro destino |
| `SEG_FILAS` | el insumo fila por fila con su `SEG_ID` y su mejor enlace |
| `DEDUP_SEG` · `DEDUP_ORBIS` · `DEDUP_FDI` | registros que son la misma entidad dentro de cada base |
| `RELACIONES_PROPIEDAD` | vínculos por dueño o matriz SIN nombre compartido (no son match) |
| `REVISION_PENDIENTE` · `DEDUP_PENDIENTE` | la cola de revisión humana: pares que el motor no decide (siglas, marca compartida, homónimos) |
| `EXCLUIDOS_REVISION` | enlaces automáticos que la revisión descartó, con la razón |
| `VERIFICACION` · `INVARIANTES` · `PARAMETROS` | la medición, los controles y la trazabilidad |

**Cómo decide.** El país es un bloqueo duro: solo se comparan empresas del mismo país,
canonizado contra el catálogo ISO 3166 de la librería (España = Spain). Dentro de cada país
se comparan **todos** los pares —las bases son pequeñas: ninguna coincidencia se pierde por
el bloqueo— y cada par se clasifica con la normalización y el comparador de la librería:

- **`misma_empresa`** — todas las palabras que no son forma legal se corresponden (se admite
  una errata de una letra por palabra) y el comparador de la librería no baja de 0,70.
  `GALLETAS GULLON, S.A.,` = `GALLETAS GULLON, SA`; `DR SCHAER SPA.` = `Dr Schar`.
- **`mismo_grupo`** — comparten una palabra distintiva y **todo** lo que sobra, en los dos
  lados, es genérico (país, sector, forma de grupo): `LIDL EXPORT INTERNATIONAL` ~ `Lidl Polska`;
  `ZENTIS SUESSWAREN` ~ `Zentis`. Si lo que sobra identifica a otra empresa, no se enlaza:
  `LUDWIG SCHOKOLADE` ≠ `LUDWIG WEINRICH`.
- **Relación de propiedad** — vínculo por dueño último o matriz sin nombre compartido
  (Picnic GmbH, cuyo dueño es EDEKA). Solo la aporta la revisión, y va aparte: no es un match.
- **Cola de revisión** — lo que el nombre no basta para decidir va a `REVISION_PENDIENTE`:
  una marca distintiva compartida con sobrantes no genéricos (`EDEKA MINDEN`), siglas y
  abreviaturas (`GR.EMP. P. ALIMENTACION` → `GRUPO EMPRESARIAL PALACIOS ALIMENTACION`) y
  nombres muy cercanos. El motor nunca acepta estos pares solo.

**Deduplicación.** ORBIS se deduplica **solo por identificador** (BvD raíz; en Francia, el
SIREN de un SIRET): dos nombres iguales con BvD distinto son personas jurídicas distintas.
SEG, por nombre (mismas palabras, erratas de una letra, **misma forma legal**) y, además, dos
grafías que son la misma empresa que un mismo registro ORBIS son la misma empresa. En FDI solo
se unen sin revisión las filas con el nombre idéntico: en fDi Markets dos nombres parecidos
suelen ser la matriz y su filial (`Aldi` / `Aldi Inc`). La deduplicación de la librería (`deduplicar_importadores`) se midió y
**se descartó** para estas bases: trata la geografía como ruido y une filiales de un grupo
(`Ferrero France` ~ `Ferrero Russia`), con precisión de 0,09 en ORBIS y 0,02 en FDI.

**Revisión adjudicada (opcional).** Con `revision_adjudicada_piloto.csv` el resultado es el
**verificado**. Cómo se construyó: (1) cada lote de empresas lo revisaron tres jueces
independientes con métodos distintos (candidatos, búsqueda propia, contexto) y cada
desacuerdo, tres árbitros escépticos; (2) un auditor de completitud recorrió, país por país,
la lista **completa** de ORBIS y FDI para cada una de las 356 empresas SEG buscando lo que se
hubiera escapado; (3) cada hallazgo, objeción o decisión por arbitraje pasó por tres
verificadores con lentes distintas (identidad legal, propiedad, abogado del diablo); los
empates se resolvieron con evidencia primaria, que queda citada en la columna `RAZON`. El
notebook aplica esas decisiones (puede **unir** y también **vetar** uniones automáticas) y
**mide** al motor automático contra ellas. Sin el archivo, el resultado es solo el automático
y las hojas `REVISION_PENDIENTE` y `DEDUP_PENDIENTE` dicen dónde mirar.

## Resultado medido en el piloto

**Resultado verificado** (con `revision_adjudicada_piloto.csv`): 485 filas de la segmentación → **329 empresas**.
De las 310 empresas en países con contraparte, **25 tienen coincidencia**: 22 en ORBIS (32 enlaces), 16 en FDI
(45 enlaces) y 13 en las dos. Esas 25 empresas suman el 32 % de las operaciones y el 28 % de los kilos de la
segmentación. Otras 41 relaciones de propiedad **sin nombre compartido** (de 11 empresas) van aparte (Kaufland frente a Lidl; Adam Foods frente a
Cuétara, Artiach y Seurodis). Las 19 empresas de Países Bajos no tienen contraparte posible.

**Motor automático, medido contra la referencia** (sin aplicar la revisión; conteos con IC 95 % de Wilson):

| | precisión | recall | recall contando la cola de revisión |
|---|---|---|---|
| SEG ↔ ORBIS | 25/25 = **1,00** (0,87–1,00) | 25/36 = **0,69** (0,53–0,82) | 34/36 = 0,94 |
| SEG ↔ FDI | 43/44 = **0,98** (0,88–1,00) | 43/45 = **0,96** (0,85–0,99) | 45/45 = 1,00 |
| dedup SEG | 10/10 = 1,00 | 10/33 = 0,30 | — |
| dedup ORBIS | 6/6 = 1,00 | 6/7 = 0,86 | 7/7 |
| dedup FDI | 1/1 = 1,00 | 1/14 = 0,07 | por diseño: FDI no se une sin revisión |

El motor casi no se equivoca cuando acepta un par. Lo que no encuentra solo lo resuelve la revisión:
- **Siglas y abreviaturas de aduana**: `G.E.P.S.A.U`, `M. G. EMP.`. Van a la cola de revisión.
- **Filiales regionales**: `EDEKA MINDEN`. También van a la cola.
- **Casos que exigen conocimiento externo**, que ninguna regla de nombres alcanza:
  - `CONFISERIE HEIDEL` es una división de Windel.
  - `G. GASTRONOMIA` es Guzmán Gastronomía, hoy Bidfood Spain.

El único error del automático es un **homónimo**. `Hijos de Rivera` en FDI tiene sede en Málaga y es un
distribuidor de otro grupo. El nombre coincide exacto con el de la cervecera de Estrella Galicia.

**Cómo se verificó la referencia:**

| paso | qué se hizo | resultado |
|---|---|---|
| 1 | 3 jueces por lote, con métodos distintos | 11 lotes |
| 2 | 3 árbitros escépticos por cada desacuerdo | 20 pares |
| 3 | Auditoría de completitud contra la lista completa de ORBIS y FDI de cada país | 356 empresas |
| 4 | 3 verificadores con lentes distintas por hallazgo, objeción o arbitraje | 66 afirmaciones |
| 5 | Barrido exhaustivo de duplicados SEG, verificado igual | 17 propuestas |

La auditoría de completitud encontró **un solo enlace nuevo por nombre** (`G. GASTRONOMIA`). Corrigió 4 decisiones:
- `Hijos de Rivera` (Málaga) pasa a `distinta`.
- `SOS Cuetara` pasa a `distinta`, porque ya no es del mismo grupo que Cuétara SLU.
- `SEDA OUTSPAN` ↔ `SEDA SOLUBLES` y ↔ `Sociedad Española de Alimentos (SEDA)` pasan a `mismo_grupo` (son dos decisiones).

Además reclasificó 20 vínculos como relación de propiedad sin nombre compartido. Hubo dos empates 3–3 entre paneles:
- `Hijos de Rivera` se resolvió con evidencia primaria, citada en `RAZON`.
- `SEDA OUTSPAN` ↔ `SOCIEDAD ESPANOLA DE ALIMENTOS` se resolvió de forma conservadora: no se afirma.

La referencia no es una verdad de campo y su error residual no está medido. `ORIGEN_REVISION` muestra los
votos de cada decisión: las que no son unánimes son las menos seguras. Por ejemplo, `C. SLU.` = `CUETARA SLU`
salió 2 contra 1.


## Antes de usarlo

- **FDI · País de consulta** es el país del inversionista consultado en fDi Markets, no la sede
  de cada filial: `Lidl Polska` figura bajo Alemania. Por eso las filiales extranjeras de un
  grupo alemán enlazan con empresas alemanas de la segmentación, como `mismo_grupo`.
- **Países Bajos** (22 filas de la segmentación) no existe en ORBIS ni en FDI: con el país
  como regla, esas empresas no pueden tener coincidencia.
- **`Valor U$S` viene en cero** en las 485 filas de la segmentación: las métricas útiles son
  `Operaciones` y `Kilos`.
- **La segmentación tiene ruido de aduana**: empresas de otros sectores con partidas de
  alimentos (`AMURRIO FERROCARRIL Y EQUIPOS`, `BORGWARNER EMISSIONS SYSTEMS SPAIN`), nombres
  truncados a 35 caracteres y grafías corruptas (`S.URODIS` por `SEURODIS`). El cruce trabaja
  con el nombre tal como viene; no corrige el código de producto.
- **Un nombre idéntico puede ser otra empresa.** `Hijos de Rivera` en FDI (Málaga) es un
  distribuidor homónimo, no la cervecera de Estrella Galicia: el motor lo enlaza por nombre y
  solo la revisión lo descarta. Sin revisión, la precisión medida del automático es 1,00 en
  ORBIS y 0,98 en FDI, no 1,00.
- **`mismo_grupo` no es la misma empresa.** Úselo para contexto de grupo (inversión, tamaño
  del grupo), no para atribuir a la empresa los datos financieros de otra entidad.

**Orden:** 1 → 2 → 3 → … → 10. Lo único que se edita es la celda **2**.

## 1 · Entorno
Monta Drive e instala `rues-linker` desde el tag publicado en GitHub. Si en el kernel había otra versión importada, pide reiniciar.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  1 · ENTORNO: Drive y rues-linker                                      ║
# ╚══════════════════════════════════════════════════════════════════════╝
import importlib
import subprocess
import sys

#  De dónde instalar la librería. Por defecto, el tag publicado en GitHub (repositorio público).
#  Alternativa: la ruta a su copia en Drive, p. ej. "/content/drive/MyDrive/ProColombia/rues_linker_pruebas".
ORIGEN_LIBRERIA = "git+https://github.com/EnriqueForero/rues-linker_v2@v0.22.4"  # @param {type:"string"}
VERSION_MINIMA = (0, 22, 4)

try:  # Colab: montar Drive
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")
except ImportError:
    print("ℹ️  fuera de Colab: no se monta Drive")


def _version_instalada() -> tuple[int, ...] | None:
    from importlib.metadata import PackageNotFoundError, version

    try:
        return tuple(int(x) for x in version("rues-linker").split(".")[:3])
    except (PackageNotFoundError, ValueError):
        return None


if (_version_instalada() or (0,)) < VERSION_MINIMA:
    if "record_linkage" in sys.modules:
        raise RuntimeError(
            "Qué pasó: hay una versión vieja de record_linkage ya importada en este kernel.\n"
            "Qué hacer: Entorno de ejecución → Reiniciar sesión, y vuelva a correr desde esta celda."
        )
    print(f"📦 instalando rues-linker desde {ORIGEN_LIBRERIA} …")
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", f"rues-linker @ {ORIGEN_LIBRERIA}"
                    if ORIGEN_LIBRERIA.startswith("git+") else ORIGEN_LIBRERIA], check=True)
    importlib.invalidate_caches()

import record_linkage  # noqa: E402

v = _version_instalada()
if v is None or v < VERSION_MINIMA:
    raise RuntimeError(f"rues-linker {v} < {VERSION_MINIMA}: la instalación no quedó bien.")
for m in ("flujo.importadores", "matching.nombre_idf", "paises.catalogo", "processing.saneamiento", "pipeline.result"):
    importlib.import_module(f"record_linkage.{m}")
print(f"✅ rues-linker {record_linkage.__version__} · {record_linkage.__file__}")

## 2 · Configuración
Rutas, hojas y columnas de cada fuente, grafías de país, listas de limpieza y umbrales. **Es la única celda que se edita.**

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  2 · CONFIGURACIÓN — LA ÚNICA CELDA QUE SE EDITA                        ║
# ╚══════════════════════════════════════════════════════════════════════╝

# ── 2.1 · Archivos ───────────────────────────────────────────────────────
CARPETA = "/content/drive/MyDrive/ProColombia/Piloto_agroalimentos"  # @param {type:"string"}
ARCHIVO_SEGMENTACION = f"{CARPETA}/Piloto_segmentacion_Empresas_exportadoras.xlsx"
ARCHIVO_DESCARGAS = f"{CARPETA}/Base_Piloto3_Descargas_Agroalimentos.xlsx"
#  Revisión adjudicada (opcional). Con el archivo, el resultado es el verificado y el
#  notebook además MIDE al motor automático contra él. "" = solo el motor automático.
ARCHIVO_REVISION = f"{CARPETA}/revision_adjudicada_piloto.csv"  # @param {type:"string"}
CARPETA_SALIDA = f"{CARPETA}/resultados"  # @param {type:"string"}
#  Sello de fecha y hora en el nombre del Excel: una corrida nueva no pisa la anterior.
VERSIONAR_SALIDA = True  # @param {type:"boolean"}

# ── 2.2 · Fuentes: hoja y columnas (el contrato de entrada) ──────────────
#  El cruce es por NOMBRE + PAÍS. Si una descarga cambia de columnas, se cambia aquí.
FUENTES = {
    "SEG": dict(archivo=ARCHIVO_SEGMENTACION, hoja=0, col_nombre="Empresa", col_pais="País de Empresa",
                cols_metricas=("Operaciones", "Valor U$S", "Kilos"), cols_contexto=("HS6", "Descripción", "Socios Comerciales"),
                largo_truncamiento=35, detectar_intermediarios=True),
    "ORBIS": dict(archivo=ARCHIVO_DESCARGAS, hoja="Orbis", col_nombre="Company name Latin alphabet", col_pais="Country",
                  col_id="BvD ID number", cols_contexto=("City Latin Alphabet", "Website address", "GUO - Name", "GUO - Country ISO code",
                  "Operating revenue (Turnover) th USD Last avail. yr", "Number of employees Last avail. yr",
                  "NACE Rev. 2, core code - description")),
    "FDI": dict(archivo=ARCHIVO_DESCARGAS, hoja="FDI", col_nombre="Company name", col_pais="País de consulta",
                cols_contexto=("Parent company", "HQ country", "HQ city (Parent)", "Website", "Primary sector", "Primary activity",
                               "No. of projects (in current search)", "Total capex USDm (in current search)", "Revenue (USDm)"),
                # fDi registra grupos: dos nombres parecidos suelen ser matriz y filial ("Aldi" / "Aldi Inc").
                # Medido: por nombre, la mitad de las uniones eran filiales. Sin revisión solo se unen
                # filas con el nombre idéntico; lo demás va a DEDUP_PENDIENTE.
                dedup_automatico=False),
}

# ── 2.3 · Países ─────────────────────────────────────────────────────────
#  Grafías en inglés que el catálogo 0.22.4 no trae (medido: 32 de 60 nombres comunes;
#  sin FRANCE e ITALY, 600 filas de ORBIS quedaban sin país). Si una descarga trae otro
#  país, la celda 5 (preparación) se detiene y lo lista: agréguelo aquí con su ISO3.
ALIAS_PAISES = {
    "FRA": ("FRANCE",), "ITA": ("ITALY",), "CHE": ("SWITZERLAND",), "BEL": ("BELGIUM",), "THA": ("THAILAND",),
    "ZAF": ("SOUTH AFRICA",), "LUX": ("LUXEMBOURG",), "POL": ("POLAND",), "SWE": ("SWEDEN",), "NOR": ("NORWAY",),
    "DNK": ("DENMARK",), "FIN": ("FINLAND",), "IRL": ("IRELAND",), "GRC": ("GREECE",), "JPN": ("JAPAN",),
    "KOR": ("SOUTH KOREA",), "CZE": ("CZECH REPUBLIC", "CZECHIA"), "HUN": ("HUNGARY",), "RUS": ("RUSSIA", "RUSSIAN FEDERATION"),
    "ARE": ("UNITED ARAB EMIRATES",), "SAU": ("SAUDI ARABIA",), "EGY": ("EGYPT",), "MAR": ("MOROCCO",),
    "PHL": ("PHILIPPINES",), "NZL": ("NEW ZEALAND",), "HRV": ("CROATIA",), "SVK": ("SLOVAKIA",), "SVN": ("SLOVENIA",),
    "LTU": ("LITHUANIA",), "LVA": ("LATVIA",), "CYP": ("CYPRUS",), "ISL": ("ICELAND",),
}

# ── 2.4 · Listas de limpieza (datos declarados, no inferidos) ─────────────
#  Formas legales europeas que el normalizador 0.22.4 no quita (medido sobre ORBIS).
FORMAS_LEGALES_EXTRA = frozenset({
    "STIFTUNG", "SE", "EG", "KGAA", "MBH", "UG", "OHG", "EV", "AKTIENGESELLSCHAFT", "GESELLSCHAFT MBH",
    "GESELLSCHAFT MIT BESCHRANKTER HAFTUNG", "SOCIETA PER AZIONI", "SOCIETA A RESPONSABILITA LIMITATA",
    "SOCIETA COOPERATIVA", "SOCIETA COOPERATIVA AGRICOLA", "IN LIQUIDAZIONE", "SOCIETE PAR ACTIONS SIMPLIFIEE",
    "SOCIETE ANONYME", "SOCIEDAD LIMITADA UNIPERSONAL", "SOCIEDAD ANONIMA UNIPERSONAL", "UNIPERSONAL",
    "SAU", "SLU", "SASU", "EURL", "SNC", "SCA", "SCARL", "A SOCIO UNICO", "CON SOCIO UNICO", "SOCIO UNICO",
    "IN SIGLA", "SIGLA", "IN FORMA ABBREVIATA", "ABBREVIATA", "MIT BESCHRAENKTER HAFTUNG", "GESELLSCHAFT MIT BESCHRAENKTER HAFTUNG",
})

#  Palabras de sector, geografía y forma de grupo: cuentan como ruido, nunca como identidad.
#  Es la palanca principal de calidad: una palabra genérica que falte aquí puede unir dos
#  empresas; una marca puesta aquí por error impide enlazar su grupo.
GENERICOS_EXTRA = frozenset({
    "SCHOKOLADE", "SCHOKOLADEN", "SCHOKOLADENFABRIK", "SCHOKOLADENMANUFAKTUR", "SUESSWAREN", "SUSSWAREN",
    "CONFISERIE", "CHOCOLATERIE", "LEBENSMITTEL", "FEINKOST", "BACKWAREN", "KAFFEE", "HANDEL", "HANDELS",
    "VERTRIEB", "VERTRIEBS", "GRUPPE", "BETEILIGUNGS", "INDUSTRIE", "FABRIK", "MANUFAKTUR",
    "DOLCIARIA", "DOLCIARIO", "DOLCIARIE", "CIOCCOLATO", "BISCOTTI", "BISCOTTIFICIO", "PASTICCERIA",
    "ALIMENTARI", "ALIMENTARE", "CAFFE", "GRUPPO", "PRODOTTI", "ITALIANA",
    "BISCUITERIE", "BISCUITS", "CHOCOLAT", "CHOCOLATS", "BOULANGERIE", "PATISSERIE", "ALIMENTAIRE",
    "ALIMENTAIRES", "GROUPE", "COMPAGNIE", "FRANCAISE",
    "GALLETAS", "PANADERIA", "PASTELERIA", "BOLLERIA", "CHOCOLATES", "DULCES", "CAFES", "ALIMENTACION",
    "ALIMENTARIA", "IBERICA", "IBERIA", "ESPANOLA", "HARINERA", "CONSERVAS", "EMPRESARIAL",
    "DEUTSCHLAND", "OSTERREICH", "SCHWEIZ", "SUISSE", "POLSKA", "NEDERLAND", "BENELUX", "BLUMEN", "FIORI", "FLEURS",
    "LIETUVA", "LATVIJA", "EESTI", "MAGYARORSZAG", "CESKA", "CESKO", "REPUBLIKA", "SLOVENSKO", "SLOVENIJA", "HRVATSKA",
    "SRBIJA", "ROMANIA", "BULGARIA", "BELGIQUE", "BELGIE", "NEDERLANDEN", "DANMARK", "SVERIGE", "NORGE", "SUOMI",
    "HELLAS", "PORTUGAL", "TURKIYE", "UKRAINE", "RUSSIA", "CYPRUS", "MALTA", "IRELAND", "UK", "USA", "CANADA",
    "ZENTRALE", "ZENTRAL", "CENTRALE", "INTERNAZIONALE", "INTERNACIONAL", "INTERNATIONALE",
    # geografía de filiales (medido: presentes en el corpus y ausentes de la librería)
    "AUSTRIA", "OESTERREICH", "BELGIUM", "SWITZERLAND", "SVIZZERA", "CROATIA", "SLOVENIA", "SWEDEN", "CZECH", "REPUBLIC",
    "CZECHIA", "SLOVAKIA", "HUNGARY", "POLAND", "SERBIA", "GREECE", "NETHERLANDS", "DENMARK", "NORWAY", "FINLAND",
    "ESTONIA", "LATVIA", "LITHUANIA", "TURKEY", "RUS", "NORDIC", "NORDICS", "SCANDINAVIA", "BALTIC", "BALTICS", "CEE",
    "DACH", "EUROPEAN", "EURO", "EUROPEENNE", "PACIFIC", "MIDDLE", "AFRICA", "LATAM",
    # sector y formato comercial
    "SUPERMARKET", "SUPERMARKETS", "SUPERMARKETI", "SUPERMERCADOS", "SUPERMERCATI", "SUPERMARCHE", "HYPERMARCHE", "MARKT",
    "RETAIL", "DISCOUNT", "FOODSERVICE", "CATERING", "CHOCOLADEFABRIK", "CHOCOLADEFABRIKEN", "CHOCOLADE", "CHOCOLATE",
    "CHOCOLATIER", "KAKAO", "CACAO", "COCOA", "BAKERY", "BAKERIES", "BACKEREI", "BAECKEREI", "PANIFICIO", "FORNO", "DOLCI",
    "DOLCERIA", "SWEETS", "CONFECTIONERY", "CANDY", "BISCUIT", "COOKIES", "PRODUITS", "PRODUKTE", "FOODSTUFF",
    "NAHRUNGSMITTEL", "ALIMENTS", "ALIMENTI", "INGREDIENTS", "INGREDIENTES", "PROCESSING", "IMPEX", "DISTRIBUCION",
    "DISTRIBUZIONE", "COMPANIA", "MANAGEMENT", "PRODUKTION", "FACTORY", "WERK", "WERKE",
    # frecuentes en el corpus y no son marcas (medido: df ≥ 6 sin estar en ninguna lista)
    "SAN", "SANTA", "DES", "LES", "GIDA", "SANAYI", "TICARET", "VE", "DAIRY", "NUTRITION", "TECHNOLOGIES", "CASH", "CARRY",
    "AGRO", "AGRAR", "BRAUEREI", "CASA", "NATURAL", "FRERES", "MAISON", "SEAFOOD", "GOURMET", "BODEGAS", "WORLD", "BIO",
    "PLANTA", "PLANTAS", "PLANT", "STABILIMENTO", "DR", "DOTT",
})

#  Agentes de carga y navieras: en SEG solo se enlazan como 'misma_empresa'.
PATRONES_INTERMEDIARIO = (r"\bAS AGENTS? OF\b", r"\bON BEHALF OF\b", r"\bDUMMY\b", r"\bKUEHNE\b", r"\bLOGISTIC",
                          r"\bSHIPPING\b", r"\bSPED\b", r"\bSPEDITION", r"\bFREIGHT\b", r"\bCARGO\b", r"\bINTERFRACHT\b",
                          r"\bTRANSPORT", r"\bFORWARD")

# ── 2.5 · Umbrales ───────────────────────────────────────────────────────
#  'misma_empresa' exige que TODAS las palabras no legales se correspondan (erratas de una
#  letra admitidas) y que el comparador de la librería no baje de este piso. Medido: en
#  todo el universo de pares, solo 3 quedan entre 0,70 y 0,84 con todas las palabras
#  alineadas, y los 2 que entran con 0,70 son verdaderos (AUDEN/AUDENS, DR SCHAER/DR SCHAR).
UMBRAL_MISMA_EMPRESA = 0.70  # @param {type:"number"}
#  Errata admitida por palabra (Levenshtein): 1 = una inserción o supresión.
MAX_EDICIONES_PALABRA = 1  # @param {type:"integer"}

print(f"📁 {CARPETA}")
print(f"🧾 revisión: {ARCHIVO_REVISION or '(ninguna: solo automático)'}")
print(f"🎯 misma_empresa ≥ {UMBRAL_MISMA_EMPRESA} con palabras alineadas · errata ≤ {MAX_EDICIONES_PALABRA} edición")

## 3 · Motor y salidas (no editar)
Todo lo que decide si dos nombres son la misma empresa viene de la librería; estas dos celdas solo agregan lo propio de un cruce entre dos bases y construyen las tablas.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  MOTOR DE VINCULACIÓN — no editar (la configuración va en la celda 2)  ║
# ╚══════════════════════════════════════════════════════════════════════╝
"""Vinculación SEG ↔ ORBIS / FDI por razón social y país, sobre rues-linker.

Todo lo que decide si dos nombres son la misma empresa viene de la librería:
saneamiento, canonización de país (catálogo ISO 3166), normalizador de razones
sociales, IDF del corpus, comparador ``SimilitudNombre`` con sus tres puertas
de contención y la exportación a Excel. Este motor solo añade lo que la
librería no trae para un CRUCE entre dos bases:

1. Alias entre paréntesis como variante del nombre ("Vicky Foods (Dulcesol)").
2. Fragmento final de nombres truncados por el campo de aduana (35 caracteres).
3. Comparación exhaustiva dentro de cada país (las bases son pequeñas: no
   hace falta bloqueo LSH, y así el recall del bloqueo es 1,0 por construcción).
4. Alineación palabra a palabra: ``misma_empresa`` exige las mismas palabras
   (erratas de una edición) y el comparador estricto de la librería como piso;
   ``mismo_grupo`` exige una palabra distintiva compartida y que todo lo que
   sobra sea genérico. Siglas, marcas compartidas y cercanos van a revisión.
5. Deduplicación por identificador (ORBIS) o por la regla estricta de nombre
   con forma legal compatible (SEG, FDI). La de la librería
   (``deduplicar_importadores``) se midió y se descartó para estas bases: une
   filiales de un mismo grupo (ver ``deduplicar``).
"""
from __future__ import annotations

import hashlib
import json
import itertools
import re
import time
from dataclasses import asdict, dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

from record_linkage.flujo import ConfigImportadores
from record_linkage.flujo.importadores import SUFIJOS_INTERNACIONALES, PaisesSinClasificar, preparar, registrar_locale
from record_linkage.matching.idf import construir_idf
from record_linkage.matching.nombre_idf import SimilitudNombre, neutralizar_genericos
from record_linkage.matching.normalizadores import normalizar_nombre
from record_linkage.paises.catalogo import CATALOGO_PAISES
from record_linkage.processing.saneamiento import ascii_mayusculas, sanear_texto, unir_iniciales
from rapidfuzz.distance import Levenshtein


# ── Especificación de fuentes y configuración ─────────────────────────────
@dataclass(frozen=True)
class Fuente:
    """Una base de entrada: archivo, hoja y columnas (el contrato de entrada)."""

    clave: str
    archivo: str
    hoja: str | int
    col_nombre: str
    col_pais: str
    col_id: str | None = None
    cols_metricas: tuple[str, ...] = ()
    cols_contexto: tuple[str, ...] = ()
    #: Ancho del campo de nombre en el sistema de origen si lo trunca (aduana: 35). None = no trunca.
    largo_truncamiento: int | None = None
    #: Marcar intermediarios logísticos (solo tiene sentido en registros de aduana).
    detectar_intermediarios: bool = False
    #: Unir duplicados por nombre sin revisión. False = solo se unen filas con el nombre
    #: IDÉNTICO (la misma cadena, repetida); lo que la regla estricta uniría va a
    #: DEDUP_PENDIENTE (útil en registros de grupos, como fDi Markets, donde «Aldi» y
    #: «Aldi Inc» son la matriz y su filial). Sin efecto si la fuente tiene ``col_id``.
    dedup_automatico: bool = True


@dataclass
class ConfigVinculacion:
    """Parámetros del cruce. Los defectos están medidos sobre este piloto."""

    alias_paises: dict[str, tuple[str, ...]] = field(default_factory=dict)
    formas_legales_extra: frozenset[str] = frozenset()
    genericos_extra: frozenset[str] = frozenset()
    patrones_intermediario: tuple[str, ...] = ()
    umbral_misma_empresa: float = 0.84
    max_dif_informativa_grupo: int = 2
    #: Dos palabras de ≥ 4 letras "son la misma" si difieren en a lo sumo estas ediciones
    #: (Levenshtein) y la edición es una inserción o supresión (SCHÄR→SCHAER/SCHAR,
    #: AUDEN/AUDENS); una sustitución solo cuenta en palabras de ≥ 8 letras. Medido en el
    #: piloto: Jaro-Winkler ≥ 0,92 unía MUSETTI (café) con MUTTI (tomate), y una
    #: sustitución corta unía ARBAL FOOD con ARGAL ALIMENTACION, dos empresas distintas.
    max_ediciones_palabra: int = 1
    geografia_es_ruido: bool = True
    usar_alias_parentesis: bool = True

    def __post_init__(self) -> None:
        if not 0.5 <= self.umbral_misma_empresa < 1.0:
            raise ValueError(f"umbral_misma_empresa={self.umbral_misma_empresa}: debe estar en [0,5, 1,0).")
        if self.max_dif_informativa_grupo < 0:
            raise ValueError("max_dif_informativa_grupo debe ser ≥ 0.")


def catalogo_con_alias(alias: dict[str, tuple[str, ...]]) -> tuple:
    """Catálogo de países de la librería + grafías declaradas (p. ej. FRANCE, ITALY)."""
    iso_validos = {iso for iso, _, _ in CATALOGO_PAISES}
    desconocidos = sorted(set(alias) - iso_validos)
    if desconocidos:
        raise KeyError(f"ALIAS_PAISES usa ISO3 que no existen en el catálogo: {desconocidos}")
    return tuple(
        (iso, nombre, tuple(grafias) + tuple(str(a).strip().upper() for a in alias.get(iso, ())))
        for iso, nombre, grafias in CATALOGO_PAISES
    )


def config_libreria(cfg: ConfigVinculacion, col_nombre: str, col_pais: str, **extra) -> ConfigImportadores:
    """``ConfigImportadores`` de la librería con el catálogo, sufijos y genéricos declarados."""
    return ConfigImportadores(
        col_razon_social=col_nombre,
        col_pais=col_pais,
        catalogo_paises=catalogo_con_alias(cfg.alias_paises),
        sufijos_extra=SUFIJOS_INTERNACIONALES | frozenset(cfg.formas_legales_extra),
        genericos_extra=frozenset(cfg.genericos_extra),
        geografia_es_ruido=cfg.geografia_es_ruido,
        locale="PILOTO_AGRO",
        verboso=False,
        **extra,
    )


# ── Lectura y contrato de entrada ─────────────────────────────────────────
def leer_fuente(f: Fuente) -> pd.DataFrame:
    """Lee la hoja como texto y verifica que existan las columnas declaradas."""
    df = pd.read_excel(f.archivo, sheet_name=f.hoja, dtype=str)
    requeridas = [f.col_nombre, f.col_pais, *([f.col_id] if f.col_id else []), *f.cols_metricas, *f.cols_contexto]
    faltan = [c for c in requeridas if c not in df.columns]
    if faltan:
        raise KeyError(
            f"Qué pasó: a la fuente {f.clave} (hoja {f.hoja!r}) le faltan las columnas {faltan}.\n"
            f"Qué hacer: corrija el perfil en la celda de configuración. Columnas disponibles: {list(df.columns)}"
        )
    df = df.reset_index(drop=True)
    df.insert(0, "FILA_EXCEL", np.arange(2, len(df) + 2))  # fila 1 = encabezado
    vacias = df.drop(columns="FILA_EXCEL").isna().all(axis=1)
    return df[~vacias].reset_index(drop=True)


def sha256_archivo(ruta: str | Path) -> str:
    h = hashlib.sha256()
    with open(ruta, "rb") as fh:
        for bloque in iter(lambda: fh.read(1 << 20), b""):
            h.update(bloque)
    return h.hexdigest()


# ── Normalización de nombres ─────────────────────────────────────────────
_PAREN = re.compile(r"\(([^()]*)\)")
_PAREN_ABIERTO = re.compile(r"\([^()]*$")


def separar_alias(nombre: str) -> tuple[str, str]:
    """"Vicky Foods (Dulcesol)" → ("Vicky Foods", "Dulcesol")."""
    if not isinstance(nombre, str) or "(" not in nombre:
        return (nombre if isinstance(nombre, str) else ""), ""
    alias = " ".join(m.strip() for m in _PAREN.findall(nombre) if m.strip())
    # Un paréntesis sin cerrar es un nombre truncado a mitad del alias: lo que queda
    # adentro no es confiable como alias ("… (PLANTA" ≠ "PLANTAS DE NAVARRA").
    principal = _PAREN_ABIERTO.sub(" ", _PAREN.sub(" ", nombre))
    return re.sub(r"\s+", " ", principal).strip(), alias


def normalizar(serie: pd.Series, lib: ConfigImportadores) -> pd.Series:
    """La misma cadena de normalización que ``preparar()`` de la librería."""
    registrar_locale(lib)
    limpio = unir_iniciales(
        ascii_mayusculas(sanear_texto(serie.astype("string").fillna(""), prelimpieza=lib.prelimpieza)),
        activo=lib.unir_iniciales_sueltas,
    )
    return normalizar_nombre(limpio, locale=lib.locale, quitar_sufijos=True, quitar_genericos=lib.quitar_genericos_del_nombre)


def _tokens_legales(cfg: ConfigVinculacion) -> set[str]:
    base = {"GMBH", "KG", "SA", "SL", "SLU", "SAU", "SRL", "SPA", "SAS", "SASU", "SARL", "BV", "NV", "AG", "SE",
            "LTD", "LIMITED", "INC", "LLC", "CO", "SOCIEDAD", "LIMITADA", "ANONIMA", "UNIPERSONAL", "SOCIETA",
            "RESPONSABILITA", "AZIONI", "SOCIETE", "ANONYME", "GESELLSCHAFT", "HAFTUNG", "BESCHRANKTER",
            "AKTIENGESELLSCHAFT", "STIFTUNG", "KGAA", "MBH", "LTDA", "STI", "COOPERATIVA", "COOPERATIVE"}
    for f in set(SUFIJOS_INTERNACIONALES) | set(cfg.formas_legales_extra):
        base |= set(str(f).upper().split())
    return {t for t in base if len(t) >= 2}


_CONECTORES = {"Y", "E", "UND", "AND", "ET", "&", "DE", "DEL", "DI", "DU", "VON", "V"}


def _palabras_utiles(nombre: str, legales: set[str]) -> list[str]:
    """Palabras que identifican: sin formas legales, conectores ni letras sueltas."""
    return [t for t in str(nombre).split() if len(t) > 1 and t not in legales and t not in _CONECTORES and not t.isdigit()]


def nucleo(nombre: str, legales: set[str], genericos: frozenset[str]) -> str:
    """Lo que identifica al nombre: sin formas legales, conectores ni palabras genéricas."""
    return " ".join(t for t in _palabras_utiles(nombre, legales) if t not in genericos and not t.isdigit())


def alinear(a: str, b: str, legales: set[str], max_ediciones: int, truncado_a: bool = False) -> tuple[list[str], list[str]]:
    """Empareja palabra a palabra dos nombres normalizados; devuelve las que sobran en cada lado.

    Igualdad de palabra: exacta; o a lo sumo ``max_ediciones`` de Levenshtein si
    ambas miden ≥ 4 letras (una errata, una transliteración SCHÄR→SCHAER); o, si
    ``a`` viene truncado, su última palabra es prefijo de una de ``b``.
    """
    ta, tb = _palabras_utiles(a, legales), _palabras_utiles(b, legales)
    libres = list(tb)
    sobra_a = []
    for k, t in enumerate(ta):
        hit = next((u for u in libres if u == t), None)
        if hit is None and len(t) >= 4:
            cand = sorted((Levenshtein.distance(t, u), u) for u in libres if misma_palabra(t, u, max_ediciones))
            hit = cand[0][1] if cand else None
        if hit is None and truncado_a and k == len(ta) - 1 and len(t) >= 3:
            hit = next((u for u in libres if u.startswith(t)), None)
        if hit is None:
            sobra_a.append(t)
        else:
            libres.remove(hit)
    return sobra_a, libres


def misma_palabra(t: str, u: str, max_ediciones: int) -> bool:
    """Errata admitida: ≤ ``max_ediciones`` y, si es sustitución, solo en palabras de ≥ 8 letras."""
    if t == u:
        return True
    if min(len(t), len(u)) < 4 or Levenshtein.distance(t, u) > max_ediciones:
        return False
    return len(t) != len(u) or min(len(t), len(u)) >= 8


def _canon_por_ediciones(tokens: list[str], max_ediciones: int) -> dict[str, int]:
    """Agrupa palabras de ≥ 4 letras a ≤ ``max_ediciones`` (componentes conexas); el resto, solas."""
    from rapidfuzz.process import cdist
    idx = {t: k for k, t in enumerate(tokens)}
    padre = list(range(len(tokens)))

    def raiz(x):
        while padre[x] != x:
            padre[x] = padre[padre[x]]; x = padre[x]
        return x
    largos = [t for t in tokens if len(t) >= 4]
    if max_ediciones > 0 and len(largos) > 1:
        d = cdist(largos, largos, scorer=Levenshtein.distance, score_cutoff=max_ediciones, dtype=np.int32, workers=-1)
        for i, j in zip(*np.nonzero(d <= max_ediciones)):
            if i < j and misma_palabra(largos[i], largos[j], max_ediciones):
                padre[raiz(idx[largos[i]])] = raiz(idx[largos[j]])
    return {t: raiz(idx[t]) for t in tokens}


def _fichas_sigla(nombre: str, legales: set[str]) -> list[str]:
    """Palabras para comparar siglas: sin formas legales ni conectores, CON letras sueltas."""
    return [t for t in str(nombre).split() if t not in legales and t not in _CONECTORES and not t.isdigit()]


def es_abreviatura(a: str, b: str, legales: set[str], genericos: frozenset[str]) -> bool:
    """¿El nombre ``a`` es una abreviatura de ``b``?  Solo alimenta la cola de revisión.

    Cada palabra de ``a``, en orden, es prefijo de una palabra de ``b`` o la sigla de
    palabras consecutivas de ``b``; la primera ancla en la primera palabra de ``b``; al
    menos una es abreviatura de verdad y cubre una palabra distintiva de ``b``. Casos
    reales de aduana (35 caracteres): «GR.EMP. P. ALIMENTACION» → GRUPO EMPRESARIAL
    PALACIOS ALIMENTACION; «M. G. EMP.» → MONBAKE GRUPO EMPRESARIAL; «G.E.P.S.A.U» →
    GRUPO EMPRESARIAL PALACIOS (+ SAU pegado). «P.» puede ser PALACIOS o PASCUAL: por eso
    un humano decide.
    """
    A, B = _fichas_sigla(a, legales), _fichas_sigla(b, legales)
    if not A or not B or A[0][0] != B[0][0]:
        return False
    variantes = [A]
    ult = A[-1]
    for leg in legales:                              # forma legal pegada: GEPSAU → GEP
        if " " not in leg and len(leg) >= 2 and ult.endswith(leg) and len(ult) - len(leg) >= 2:
            variantes.append(A[:-1] + [ult[: -len(leg)]])

    def cubre(A, i, j, abrevio, distintiva):
        if i == len(A):
            return abrevio and distintiva
        if j >= len(B):
            return False
        t = A[i]
        if B[j].startswith(t) and cubre(A, i + 1, j + 1, abrevio or len(t) < len(B[j]),
                                         distintiva or (len(t) < len(B[j]) and B[j] not in genericos)):
            return True
        n = len(t)
        if 2 <= n <= len(B) - j and all(B[j + k][0] == t[k] for k in range(n)):
            if cubre(A, i + 1, j + n, True, distintiva or any(x not in genericos for x in B[j:j + n])):
                return True
        return i > 0 and cubre(A, i, j + 1, abrevio, distintiva)

    return any((len(v) >= 2 or len(v[0]) >= 3) and cubre(v, 0, 0, False, False) for v in variantes)


def recortar_truncado(norm: str, crudo: str, largo: int, legales: set[str]) -> str:
    """Quita del final lo que es un fragmento de forma legal cortado por el campo.

    Solo actúa si el nombre crudo ocupa el campo entero (``largo`` caracteres):
    "NETS2WORLD UNIPER" → "NETS2WORLD" (UNIPER…SONAL), "… PRALINEN GMB" →
    "… PRALINEN", "RESONUS NEW CHANNELS UND K" → "RESONUS NEW CHANNELS".
    Un fragmento que no es forma legal ("WILHELM REUSS LEBENSM") se conserva.
    """
    if not norm or len(str(crudo).rstrip()) < largo:
        return norm
    toks = norm.split()
    while len(toks) > 1:
        t = toks[-1]
        es_frag_legal = any(l.startswith(t) and l != t for l in legales) or t in legales
        if len(t) <= 1 or es_frag_legal or t in _CONECTORES:
            toks.pop()
            continue
        break
    return " ".join(toks)


def preparar_fuente(df: pd.DataFrame, f: Fuente, cfg: ConfigVinculacion, patrones_inter: re.Pattern | None) -> pd.DataFrame:
    """País canónico y nombre normalizado con la librería; alias y truncamiento aparte."""
    out = df.copy()
    crudo = out[f.col_nombre].fillna("").astype(str)
    partes = crudo.map(separar_alias) if cfg.usar_alias_parentesis else crudo.map(lambda x: (x, ""))
    out["_NOMBRE_PRINCIPAL"] = partes.str[0]
    sin_pais = out[f.col_pais].fillna("").astype(str).str.strip() == ""
    if sin_pais.any():
        raise ValueError(f"Qué pasó: {int(sin_pais.sum())} filas de {f.clave} no traen '{f.col_pais}' "
                         f"(filas Excel {out.loc[sin_pais, 'FILA_EXCEL'].tolist()[:15]}).\n"
                         "Qué hacer: el país es la regla de cruce; complételo o borre esas filas del archivo.")
    lib = config_libreria(cfg, "_NOMBRE_PRINCIPAL", f.col_pais)
    try:
        prep = preparar(out, lib)  # detiene si hay grafías de país fuera del catálogo
    except PaisesSinClasificar as e:
        ejemplos = ", ".join(f"'ISO3': ({v.upper()!r},)" for v in e.tabla.valor.head(5))
        raise ValueError(f"Qué pasó: {f.clave} trae grafías de país que el catálogo no reconoce:\n"
                         f"{e.tabla.to_string(index=False)}\n"
                         f"Qué hacer: agréguelas en ALIAS_PAISES (celda 2) con su código ISO3, p. ej. {ejemplos}, "
                         "y vuelva a correr desde la celda 4. La columna 'sugerencia' es una ayuda, no una regla.") from None
    legales = _tokens_legales(cfg)
    if f.largo_truncamiento:
        prep["NOMBRE_NORM"] = [recortar_truncado(n, c, f.largo_truncamiento, legales) for n, c in zip(prep.NOMBRE_NORM, crudo)]
    alias = normalizar(partes.str[1].reset_index(drop=True), lib).to_numpy()
    # Un alias que solo trae forma legal ("(GMBH & CO.)") no es un nombre: medido en el piloto,
    # "FR. MEYER'S SOHN (GMBH & CO.)" quedaba igual a "HUEPEDEN & CO. (GMBH & CO.)".
    util = np.array([bool(_palabras_utiles(a, legales)) for a in alias])
    prep["ALIAS_NORM"] = np.where(util & (alias != prep.NOMBRE_NORM.to_numpy()), alias, "")
    prep["TRUNCADO"] = (crudo.str.rstrip().str.len() >= f.largo_truncamiento) if f.largo_truncamiento else False
    prep["INTERMEDIARIO"] = (crudo.str.contains(patrones_inter, na=False)
                             if (f.detectar_intermediarios and patrones_inter is not None) else False)
    prep["NOMBRE_ORIGINAL"] = crudo
    prep["PAIS_ORIGINAL"] = out[f.col_pais]
    if f.col_id:
        ids = out[f.col_id].fillna("").astype(str).str.strip()
        if (ids == "").any():
            raise ValueError(f"Qué pasó: {int((ids == '').sum())} filas de {f.clave} no traen '{f.col_id}' "
                             f"(filas Excel {out.loc[ids == '', 'FILA_EXCEL'].tolist()[:15]}).\n"
                             "Qué hacer: suelen ser filas de continuación de campos multivalor de Orbis. Exporte una "
                             "fila por empresa o borre esas filas.")
        prep["ID_REGISTRO"] = ids.to_numpy()
    else:
        prep["ID_REGISTRO"] = (f.clave + "-" + out["FILA_EXCEL"].astype(str)).to_numpy()
    if prep["ID_REGISTRO"].duplicated().any():
        dup = prep.loc[prep["ID_REGISTRO"].duplicated(keep=False), "ID_REGISTRO"].unique()[:5]
        raise ValueError(f"{f.clave}: el identificador {f.col_id!r} no es único (p. ej. {list(dup)}).")
    return prep


# ── Deduplicación interna ─────────────────────────────────────────────────
# Formas legales por familia: dos registros con formas de familias distintas son
# personas jurídicas distintas (SL ≠ S. COOP.; AG ≠ GMBH; STIFTUNG & CO. KG ≠ GMBH & CO. KG).
# SL/SLU o SA/SAU son la misma familia: la aduana escribe la misma empresa de las dos maneras.
_FAMILIAS_LEGALES = [
    ("STIFTUNG_KG", r"STIFTUNG (?:&|\+|UND) CO KG"),
    ("SE_KG", r"\bSE (?:&|\+|UND) CO KG"),
    ("GMBH_KG", r"(?:GMBH|AG) (?:&|\+|UND) CO (?:KG|OHG)"),
    ("KGAA", r"\bKGAA\b"),
    ("OHG", r"\bOHG\b"),
    ("KG", r"\bKG\b"),
    ("GMBH", r"\bGMBH\b|BESCHRAENKTER HAFTUNG"),
    ("AG", r"\bAG\b|AKTIENGESELLSCHAFT"),
    ("SE", r"\bSE\b"),
    ("COOP", r"\bS ?COOP\b|COOPERATIVA|\bEG\b|GENOSSENSCHAFT|\bSAT\b"),
    ("SAS", r"\bSASU?\b"),
    ("SL", r"\bSLU?\b|\bSRLS?\b|\bSARL\b|\bEURL\b|\bKFT\b|\bSP ZOO\b|LIMITADA|RESPONSABILITA LIMITATA"),
    ("SCA", r"\bSCA\b"),
    ("SA", r"\bSAU?\b|\bSPA\b|ANONIMA|PER AZIONI"),
    ("BV", r"\bBV\b"), ("NV", r"\bNV\b"),
    ("LTD", r"\bLTD\b|LIMITED\b"), ("INC", r"\bINC\b|\bCORP\b"), ("LLC", r"\bLLC\b"),
]
_FAMILIAS_RE = [(k, re.compile(p)) for k, p in _FAMILIAS_LEGALES]


def formas_legales(nombres: pd.Series) -> np.ndarray:
    """Familia de la forma legal escrita en cada nombre original ('' si no hay o no se reconoce)."""
    x = ascii_mayusculas(sanear_texto(nombres.astype(str)))
    x = x.str.replace(r"\([^()]*\)?", " ", regex=True)      # lo que va entre paréntesis no es la forma
    x = x.str.replace(r"[.,;:/\-]", " ", regex=True).str.replace(r"\s+", " ", regex=True)
    x = x.str.replace(r"\b([A-Z]) (?=[A-Z]\b)", r"\1", regex=True).str.strip()   # S A U → SAU

    def familia(v: str) -> str:
        for k, rx in _FAMILIAS_RE:
            if rx.search(v):
                return k
        return ""
    return np.array([familia(v) for v in x], dtype=object)


def raiz_identificador(x: str) -> str:
    """Raíz de un BvD ID: sin el sufijo de sucursal (DE…-1003) y, en Francia, el SIREN de un SIRET."""
    x = str(x).split("-")[0].strip()
    m = re.fullmatch(r"FR(\d{9})\d{5}", x)
    return f"FR{m.group(1)}" if m else x


def deduplicar(prep: pd.DataFrame, f: Fuente, cfg: ConfigVinculacion,
               comps: tuple[SimilitudNombre, SimilitudNombre, SimilitudNombre]) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """Registros de la MISMA entidad dentro de una fuente → (prep con GRUPO_DEDUP, uniones, pendientes).

    Medido contra la referencia adjudicada de este piloto (678 pares), la deduplicación de
    la librería (``deduplicar_importadores``) NO sirve para estas bases: trata la geografía
    como ruido y une filiales de un mismo grupo (Ferrero France ~ Ferrero Russia; LIDL
    STIFTUNG ~ LIDL BULGARIA). Precisión 0,09 en ORBIS y 0,02 en FDI. Por eso:

    * Fuente con identificador (ORBIS): SOLO el identificador. Mismo BvD raíz (DE…-1003 es
      una sucursal) o mismo SIREN en Francia. Dos nombres iguales con BvD distinto son
      personas jurídicas distintas (medido: HARIBO, FROSTKRONE, LIDL GMBH & CO. KG) y van a
      pendientes, no se unen. Medido: P = 1,00, R = 0,86.
    * Fuente sin identificador (SEG, FDI): la regla estricta de «misma_empresa» del motor
      (mismas palabras, erratas de una edición) sobre el nombre principal (un paréntesis
      suele ser una marca: TARBAL FOOD SLU (CONFILETAS) ≠ CONFILETAS SL), con la MISMA
      familia de forma legal en los dos (SL ≠ S. COOP.; «Aldi» ≠ «Aldi Inc», la filial) y sin
      un país o sector entre paréntesis en uno solo («Compagnie Fruitiere (UK)»).
    Lo que casi cumple va a ``pendientes`` con su motivo: lo decide un humano.
    """
    n = len(prep)
    union, pend = [], []
    formas = formas_legales(prep.NOMBRE_ORIGINAL)
    if f.col_id:
        raiz = prep.ID_REGISTRO.map(raiz_identificador).to_numpy()
        for _, idx in pd.Series(np.arange(n)).groupby(raiz):
            idx = list(idx)
            union += [(idx[0], j, "mismo identificador raíz") for j in idx[1:]]
        # homónimos con identificador distinto: se informan, no se unen
        for _, idx in pd.Series(np.arange(n)).groupby([prep.PAIS_ISO3.to_numpy(), prep.NOMBRE_NORM.to_numpy()]):
            idx = [k for k in idx if prep.NOMBRE_NORM.iat[k]]
            pend += [(a, b, "mismo nombre, identificador distinto") for a, b in itertools.combinations(idx, 2) if raiz[a] != raiz[b]]
    else:
        # misma clave natural (país + nombre tal cual): la misma empresa repetida (en SEG, una fila por
        # producto). Vale también en FDI: es la misma cadena, no una matriz y su filial. Las filas SIN
        # nombre no se unen entre sí: cada una queda como su propia entidad.
        con_nombre = prep.NOMBRE_ORIGINAL.fillna("").astype(str).str.strip() != ""
        for _, idx in pd.Series(np.arange(n))[con_nombre.to_numpy()].groupby(
                [prep.PAIS_ORIGINAL.astype(str).to_numpy()[con_nombre.to_numpy()],
                 prep.NOMBRE_ORIGINAL.astype(str).str.strip().str.replace(r"\s+", " ", regex=True).to_numpy()[con_nombre.to_numpy()]]):
            idx = list(idx)
            union += [(idx[0], j, "mismo nombre y país") for j in idx[1:]]
        legales = _tokens_legales(cfg)
        genericos = config_libreria(cfg, "x", "y").genericos_todos
        alias = prep.ALIAS_NORM.fillna("").to_numpy()
        # paréntesis que solo dice un país o un sector: «Compagnie Fruitiere (UK)» es la filial
        solo_generico = np.array([bool(x) and not nucleo(x, legales, genericos) for x in alias])
        # mismas letras sin espacios ni signos: «GBfoods» = «GB Foods», «Hub.Cycle» = «Hubcycle»
        pegado = prep.NOMBRE_NORM.fillna("").str.replace(r"[^A-Z0-9]", "", regex=True).to_numpy()
        for _, idx in pd.Series(np.arange(n)).groupby([prep.PAIS_ISO3.to_numpy(), pegado]):
            idx = [k for k in idx if pegado[k]]
            for a, b in itertools.combinations(idx, 2):
                if prep.NOMBRE_NORM.iat[a] != prep.NOMBRE_NORM.iat[b]:
                    if f.dedup_automatico and formas[a] == formas[b]:
                        union.append((a, b, "mismas letras, otros espacios"))
                    else:
                        pend.append((a, b, "mismas letras, otros espacios"))
        # Filas con el mismo nombre ya quedaron unidas arriba: se compara una fila por nombre
        # (en SEG, una por grafía y no una por producto) y el resultado vale para todas.
        rep = np.flatnonzero(~pd.Series(list(zip(prep.PAIS_ORIGINAL.astype(str), prep.NOMBRE_ORIGINAL.astype(str)))).duplicated().to_numpy())
        pr = prep.iloc[rep].reset_index(drop=True)
        q = _vincular(pr, pr, comps, cfg)
        q["pos_seg"], q["pos_dest"] = rep[q.pos_seg.to_numpy().astype(int)], rep[q.pos_dest.to_numpy().astype(int)]
        q = q[q.pos_seg != q.pos_dest].copy()
        # Cada par sin orden se evalúa en los dos sentidos (el truncado solo se aplica al lado
        # izquierdo) y se queda el mejor: misma_empresa > mismo_grupo > sin clase.
        q["a"], q["b"] = np.minimum(q.pos_seg, q.pos_dest), np.maximum(q.pos_seg, q.pos_dest)
        q["_r"] = q.CLASE.map({"misma_empresa": 0, "mismo_grupo": 1}).fillna(2)
        q = q.sort_values(["_r", "SIM_ESTRICTA"], ascending=[True, False]).drop_duplicates(["a", "b"])
        for a, b, clase, por_alias, abrev, sa, sb, sim in zip(q.a, q.b, q.CLASE, q.POR_ALIAS, q.ABREVIATURA,
                                                             q.SOBRA_SEG, q.SOBRA_DEST, q.SIM_ESTRICTA):
            fa, fb = formas[a], formas[b]
            if clase == "misma_empresa":
                if not f.dedup_automatico:
                    pend.append((a, b, "mismo nombre (esta fuente no une sin revisión)"))
                elif por_alias:
                    pend.append((a, b, "mismo nombre solo por el paréntesis (¿marca?)"))
                elif fa != fb:
                    pend.append((a, b, f"mismo nombre, forma legal distinta ({fa or '—'} ≠ {fb or '—'})"))
                elif alias[a] != alias[b] and (solo_generico[a] or solo_generico[b]):
                    pend.append((a, b, "mismo nombre, uno lleva un país o sector entre paréntesis"))
                else:
                    union.append((a, b, "mismo nombre (regla estricta)"))
            elif abrev:
                pend.append((a, b, "abreviatura o sigla"))
            elif not f.dedup_automatico:
                # En un registro de grupos, contener al otro nombre o parecérsele es ser filial
                # (Lidl ~ Lidl Polska; Ferrero France ~ Ferrero Russia): no es un pendiente.
                continue
            elif clase == "mismo_grupo" and (not sa or not sb) and fa == fb:
                pend.append((a, b, "un nombre contiene al otro y lo que sobra es genérico"))
            elif sim >= 0.80:
                pend.append((a, b, f"nombre muy parecido ({sim:.2f})"))
    tipos = {"pos_a": "int64", "pos_b": "int64"}
    aristas = pd.DataFrame(union, columns=["pos_a", "pos_b", "REGLA"]).astype(tipos).drop_duplicates(["pos_a", "pos_b"])
    pendientes = pd.DataFrame(pend, columns=["pos_a", "pos_b", "MOTIVO"]).astype(tipos).drop_duplicates(["pos_a", "pos_b"])
    # Pendientes que ya quedaron unidos por otra vía no son pendientes
    out = prep.copy()
    grupo = _componentes(n, aristas.pos_a, aristas.pos_b)
    out["GRUPO_DEDUP"] = [f"{f.clave}-G{g:04d}" for g in grupo]
    out["N_EN_GRUPO"] = out.groupby("GRUPO_DEDUP").GRUPO_DEDUP.transform("size")
    out["FORMA_LEGAL"] = formas
    if len(pendientes):
        pendientes = pendientes[grupo[pendientes.pos_a.to_numpy()] != grupo[pendientes.pos_b.to_numpy()]]
    return out, aristas, pendientes.reset_index(drop=True)


def _componentes(n: int, a, b) -> np.ndarray:
    """Componentes conexas (union-find) numeradas 1..k por orden de primera aparición."""
    padre = list(range(n))

    def raiz(x):
        while padre[x] != x:
            padre[x] = padre[padre[x]]
            x = padre[x]
        return x
    for x, y in zip(a, b):
        rx, ry = raiz(int(x)), raiz(int(y))
        if rx != ry:
            padre[max(rx, ry)] = min(rx, ry)
    r = np.array([raiz(i) for i in range(n)])
    return pd.factorize(r)[0] + 1


# ── Vinculación ──────────────────────────────────────────────────────────
def construir_comparadores(corpus: pd.Series, cfg: ConfigVinculacion) -> tuple[SimilitudNombre, SimilitudNombre, SimilitudNombre]:
    """Tres comparadores de la librería sobre el mismo corpus.

    - estricto: defectos de la librería (decide ``misma_empresa``).
    - grupo: la puerta de contención relajada a ``max_dif_informativa_grupo``.
    - núcleo: el de grupo, pero sobre el nombre sin genéricos ni formas legales,
      con su propio IDF. Existe porque Jaro-Winkler pesa el 75 % y castiga un
      sufijo distinto aunque sea genérico: medido, "LIDL EXPORT INTERNATIONAL"
      frente a "LIDL POLSKA" da 0,73 con el comparador completo y 1,0 en núcleo.
    """
    lib = config_libreria(cfg, "x", "y")
    legales = _tokens_legales(cfg)
    vocab = pd.Series(sorted(set(corpus) - {""}))
    pesos = neutralizar_genericos(construir_idf(vocab.reset_index(drop=True)), lib.genericos_todos, peso=lib.peso_token_generico)
    nucleos = pd.Series(sorted({nucleo(n, legales, lib.genericos_todos) for n in vocab} - {""}))
    pesos_n = construir_idf(nucleos.reset_index(drop=True))
    comun = dict(genericos_estructurales=lib.genericos_estructurales, alfa=lib.alfa_idf, prefix_weight=lib.prefix_weight,
                 min_informativos_compartidos=lib.min_informativos_compartidos, fraccion_max_distintivo=lib.fraccion_max_distintivo,
                 longitud_minima_token=lib.longitud_minima_token, ignorar_numericos=lib.ignorar_numericos)
    estricto = SimilitudNombre(pesos, vocab.to_numpy(), max_diferencia_informativa=lib.max_diferencia_informativa, **comun)
    grupo = SimilitudNombre(pesos, vocab.to_numpy(), max_diferencia_informativa=cfg.max_dif_informativa_grupo, **comun)
    nuc = SimilitudNombre(pesos_n, nucleos.to_numpy(), max_diferencia_informativa=cfg.max_dif_informativa_grupo, **comun)
    return estricto, grupo, nuc


def _variantes(prep: pd.DataFrame) -> pd.DataFrame:
    v = pd.concat([
        pd.DataFrame({"pos": np.arange(len(prep)), "PAIS_ISO3": prep.PAIS_ISO3.to_numpy(), "VAR": prep.NOMBRE_NORM.to_numpy(), "ES_ALIAS": False}),
        pd.DataFrame({"pos": np.arange(len(prep)), "PAIS_ISO3": prep.PAIS_ISO3.to_numpy(), "VAR": prep.ALIAS_NORM.to_numpy(), "ES_ALIAS": True}),
    ])
    return v[v.VAR != ""].drop_duplicates(["pos", "VAR"])


COLUMNAS_PARES = ["pos_seg", "pos_dest", "VAR_SEG", "VAR_DEST", "POR_ALIAS", "SIM_ESTRICTA", "SIM_GRUPO", "SIM_NUCLEO",
                  "SOBRA_SEG", "SOBRA_DEST", "CLASE", "REGLA", "COMPARTE_DISTINTIVA", "ABREVIATURA", "SCORE"]
#: Tope de pares por corrida: por encima, Colab se queda sin memoria (medido: ~1,6 GB por 3 millones).
MAX_PARES = 20_000_000


def _firmas(df: pd.DataFrame, cols: list[str]) -> tuple[np.ndarray, np.ndarray]:
    """(índice del representante de cada firma, firma de cada fila): lo que decide un par depende solo de esto."""
    f = pd.Series(pd.factorize(pd.MultiIndex.from_frame(df[cols].astype(str)))[0])
    return np.flatnonzero(~f.duplicated().to_numpy()), f.to_numpy()


def vincular(seg: pd.DataFrame, destino: pd.DataFrame, comps: tuple[SimilitudNombre, SimilitudNombre], cfg: ConfigVinculacion) -> pd.DataFrame:
    """Todos los pares SEG × destino del MISMO país, puntuados y clasificados.

    Exhaustivo dentro de cada país: ninguna coincidencia queda fuera por el bloqueo. Se
    calcula una vez por FIRMA distinta (país, nombre, alias y, en SEG, truncado e
    intermediario) y se reparte a las filas que la comparten: la segmentación repite la
    misma grafía en una fila por producto.
    """
    ra, fa = _firmas(seg, ["PAIS_ISO3", "NOMBRE_NORM", "ALIAS_NORM", "TRUNCADO", "INTERMEDIARIO"])
    rb, fb = _firmas(destino, ["PAIS_ISO3", "NOMBRE_NORM", "ALIAS_NORM"])
    if len(ra) < len(seg) or len(rb) < len(destino):
        p = _vincular(seg.iloc[ra].reset_index(drop=True), destino.iloc[rb].reset_index(drop=True), comps, cfg)
        filas_a = pd.Series(np.arange(len(seg))).groupby(fa).apply(list)
        filas_b = pd.Series(np.arange(len(destino))).groupby(fb).apply(list)
        p["pos_seg"] = [filas_a[fa[ra[i]]] for i in p.pos_seg]
        p["pos_dest"] = [filas_b[fb[rb[i]]] for i in p.pos_dest]
        return p.explode("pos_seg").explode("pos_dest").astype({"pos_seg": "int64", "pos_dest": "int64"}).reset_index(drop=True)
    return _vincular(seg, destino, comps, cfg)


def _vincular(seg: pd.DataFrame, destino: pd.DataFrame, comps: tuple[SimilitudNombre, SimilitudNombre], cfg: ConfigVinculacion) -> pd.DataFrame:
    estricto, grupo, comp_nucleo = comps
    va, vb = _variantes(seg), _variantes(destino)
    n_pares = sum(int((va.PAIS_ISO3 == iso).sum()) * n for iso, n in vb.PAIS_ISO3.value_counts().items())
    if n_pares > MAX_PARES:
        raise MemoryError(f"Qué pasó: la comparación exhaustiva tendría {n_pares:,} pares (tope {MAX_PARES:,}).\n"
                          "Qué hacer: divida la corrida por país o por sector, o suba MAX_PARES si tiene memoria (Colab Pro).")
    piezas = []
    for iso, a in va.groupby("PAIS_ISO3"):
        b = vb[vb.PAIS_ISO3 == iso]
        if b.empty:
            continue
        ia, ib = np.meshgrid(np.arange(len(a)), np.arange(len(b)), indexing="ij")
        ia, ib = ia.ravel(), ib.ravel()
        na, nb = a.VAR.to_numpy()[ia], b.VAR.to_numpy()[ib]
        piezas.append(pd.DataFrame({
            "pos_seg": a.pos.to_numpy()[ia], "pos_dest": b.pos.to_numpy()[ib],
            "VAR_SEG": na, "VAR_DEST": nb, "POR_ALIAS": a.ES_ALIAS.to_numpy()[ia] | b.ES_ALIAS.to_numpy()[ib],
            "SIM_ESTRICTA": (estricto.compare(na, nb) + 1) / 2, "SIM_GRUPO": (grupo.compare(na, nb) + 1) / 2,
        }))
    if not piezas:
        return pd.DataFrame(columns=COLUMNAS_PARES).astype({"pos_seg": "int64", "pos_dest": "int64", "CLASE": str,
                                                             "COMPARTE_DISTINTIVA": bool, "ABREVIATURA": bool, "POR_ALIAS": bool})
    p = pd.concat(piezas, ignore_index=True)
    # Mejor variante por par de registros (el alias compite con el nombre principal)
    p = p.sort_values(["SIM_ESTRICTA", "SIM_GRUPO"], ascending=False).drop_duplicates(["pos_seg", "pos_dest"])
    p = p.reset_index(drop=True)
    legales = _tokens_legales(cfg)
    lib = config_libreria(cfg, "x", "y")
    genericos = lib.genericos_todos
    trunc = seg.TRUNCADO.to_numpy()[p.pos_seg.to_numpy()]
    inter = seg.INTERMEDIARIO.to_numpy()[p.pos_seg.to_numpy()]
    # Palabra distintiva: no genérica, no forma legal, ≥ 3 letras. SIN tope de frecuencia: en
    # un cruce entre bases las palabras frecuentes no genéricas son las MARCAS de los grupos
    # grandes (medido: LIDL 22, DANONE 17, METRO 14, NESTLE 8), justo lo que debe enlazar.
    def distintivas(n):
        return {t for t in _palabras_utiles(n, legales) if t not in genericos and len(t) >= 3}
    dist = {n: distintivas(n) for n in pd.unique(np.concatenate([p.VAR_SEG.to_numpy(), p.VAR_DEST.to_numpy()]))}
    # Prefiltro: comparten una distintiva, igual o a ≤ max_ediciones (SCHAER/SCHAR). Las
    # palabras a una edición se agrupan una sola vez (cdist en C) y se comparan por grupo.
    canon = _canon_por_ediciones(sorted({t for v in dist.values() for t in v}), cfg.max_ediciones_palabra)
    dist_c = {n: {canon[t] for t in v} for n, v in dist.items()}
    comparten = np.array([bool(dist_c[a] & dist_c[b]) for a, b in zip(p.VAR_SEG, p.VAR_DEST)])
    mira = ((p.SIM_ESTRICTA >= 0.6) | (p.SIM_GRUPO >= 0.6)).to_numpy() | comparten
    idx = np.flatnonzero(mira)
    na = np.array([nucleo(x, legales, genericos) for x in p.VAR_SEG.to_numpy()[idx]], dtype=object)
    nb = np.array([nucleo(x, legales, genericos) for x in p.VAR_DEST.to_numpy()[idx]], dtype=object)
    sim_n = np.zeros(len(p))
    ok = (na != "") & (nb != "")
    if ok.any():
        sim_n[idx[ok]] = (comp_nucleo.compare(na[ok], nb[ok]) + 1) / 2
    p["SIM_NUCLEO"] = sim_n
    sobra_s = np.full(len(p), "", dtype=object); sobra_d = np.full(len(p), "", dtype=object)
    clase = np.full(len(p), "", dtype=object); regla = np.full(len(p), "", dtype=object)
    marca = np.zeros(len(p), dtype=bool)

    def marcas(n, truncado):
        """Iniciales sueltas y números: 'J. MARTINEZ' ≠ 'A. MARTINEZ'; 'BIMBO 2' ≠ 'BIMBO 3'."""
        t = [x for x in str(n).split() if x not in legales and x not in _CONECTORES]
        if truncado and t and len(t[-1]) == 1:
            t = t[:-1]                        # la última letra de un nombre truncado es un fragmento
        return sorted(x for x in t if (len(x) == 1 and x.isalpha()) or x.isdigit())
    for k in idx:
        a, b = p.at[k, "VAR_SEG"], p.at[k, "VAR_DEST"]
        sa, sb = alinear(a, b, legales, cfg.max_ediciones_palabra, bool(trunc[k]))
        sobra_s[k], sobra_d[k] = " ".join(sa), " ".join(sb)
        ta, tb = dist[a], dist[b]
        # Comparten una distintiva: la misma palabra, o una emparejada por la alineación
        # (erratas: SCHAER/SCHAR) — una distintiva de A que no quedó sobrando.
        comp_k = bool(ta & tb) or bool(ta - set(sa)) and bool(tb - set(sb))
        if not comp_k:
            continue
        marca[k] = True
        if not sa and not sb:
            # Las mismas palabras (erratas admitidas por la alineación), las mismas iniciales y
            # números, y el comparador de la librería como piso: misma empresa.
            if (p.at[k, "SIM_ESTRICTA"] >= cfg.umbral_misma_empresa
                    and marcas(a, bool(trunc[k])) == marcas(b, False)):
                clase[k], regla[k] = "misma_empresa", "mismas palabras"
        elif not inter[k]:
            # Mismo grupo: comparten la palabra distintiva y TODO lo que sobra, en los dos lados,
            # es genérico: país, sector, forma de grupo ("ZENTIS" ~ "ZENTIS SUESSWAREN",
            # "LIDL EXPORT INTERNATIONAL" ~ "LIDL POLSKA", "TOUTON" ~ "COCOA TOUTON PROCESSING").
            # Un sobrante que identifica deja el par fuera, aunque un nombre contenga al otro:
            # medido, "REKA MARTA FUCHS" (una persona) ⊃ "FUCHS"; "LUDWIG SCHOKOLADE" vs
            # "LUDWIG WEINRICH" (otro apellido). Esos van a REVISION_PENDIENTE.
            if all(t in genericos for t in sa + sb):
                clase[k], regla[k] = "mismo_grupo", "sobrantes genéricos"
    p["SOBRA_SEG"], p["SOBRA_DEST"], p["CLASE"], p["REGLA"] = sobra_s, sobra_d, clase, regla
    p["COMPARTE_DISTINTIVA"] = marca
    # Siglas y abreviaturas (en los dos sentidos): no deciden nada, van a REVISION_PENDIENTE.
    ab = {}
    def abrevia(x, y):
        if (x, y) not in ab:
            ab[(x, y)] = es_abreviatura(x, y, legales, genericos) or es_abreviatura(y, x, legales, genericos)
        return ab[(x, y)]
    p["ABREVIATURA"] = [abrevia(x, y) for x, y in zip(p.VAR_SEG, p.VAR_DEST)]
    p["SCORE"] = np.where(p.CLASE == "misma_empresa", p.SIM_ESTRICTA, p.SIM_NUCLEO)
    return p

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  SALIDAS, REVISIÓN E INVARIANTES — no editar                           ║
# ╚══════════════════════════════════════════════════════════════════════╝
"""Del par puntuado a las tablas que se entregan, con la revisión aplicada y verificada."""
from __future__ import annotations

import json
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

from record_linkage.pipeline.result import PipelineResult

COLUMNAS_REVISION = ["TIPO", "FUENTE", "CLAVE_A", "NOMBRE_A", "CLAVE_B", "NOMBRE_B", "DECISION", "ORIGEN_REVISION", "RAZON"]
DECISIONES_CRUCE = {"misma_empresa", "mismo_grupo", "relacion_propiedad", "distinta"}
DECISIONES_DUP = {"misma_entidad", "mismo_grupo", "distinta"}


def _clave(x: pd.Series) -> pd.Series:
    """Clave natural sin espacios de más: sobrevive a un CSV abierto y guardado en Excel."""
    return x.fillna("").astype(str).str.strip().str.replace(r"\s+", " ", regex=True)


def clave_seg(prep: pd.DataFrame) -> np.ndarray:
    """Clave natural de una fila SEG: «País de Empresa|Empresa» como vienen en el Excel (espacios normalizados)."""
    return (_clave(prep.PAIS_ORIGINAL) + "|" + _clave(prep.NOMBRE_ORIGINAL)).to_numpy()


def leer_revision(ruta: str | None) -> pd.DataFrame | None:
    """Lee el archivo de revisión (opcional) y valida su contrato."""
    if not ruta:
        return None
    p = Path(ruta)
    if not p.exists():
        print(f"  ℹ️  sin archivo de revisión en {p} → resultado solo automático")
        return None
    # Excel en español guarda los CSV con «;» y, si no es «CSV UTF-8», en cp1252.
    try:
        texto = p.read_text(encoding="utf-8-sig"); cod = "utf-8-sig"
    except UnicodeDecodeError:
        texto = p.read_text(encoding="cp1252"); cod = "cp1252"
    cabecera = texto.splitlines()[0] if texto else ""
    sep = ";" if cabecera.count(";") > cabecera.count(",") else ","
    try:
        r = pd.read_csv(p, dtype=str, keep_default_na=False, encoding=cod, sep=sep)
    except pd.errors.ParserError as e:
        raise ValueError(f"Qué pasó: no se pudo leer {p.name} como CSV (separador {sep!r}, codificación {cod}): {e}\n"
                         "Qué hacer: guárdelo desde Excel como «CSV UTF-8 (delimitado por comas)».") from None
    r.columns = [c.strip() for c in r.columns]
    r = r.apply(lambda c: c.str.strip())
    r = r[(r != "").any(axis=1)].reset_index(drop=True)
    faltan = [c for c in COLUMNAS_REVISION if c not in r.columns]
    if faltan:
        raise KeyError(f"Qué pasó: el archivo de revisión no trae las columnas {faltan} (separador leído: {sep!r}).\n"
                       f"Qué hacer: guárdelo como CSV UTF-8 con las columnas {COLUMNAS_REVISION}.")
    for c in ("TIPO", "FUENTE", "DECISION"):
        r[c] = r[c].str.strip().str.lower() if c != "FUENTE" else r[c].str.strip().str.upper()
    r["CLAVE_A"], r["CLAVE_B"] = _clave(r.CLAVE_A), _clave(r.CLAVE_B)
    malas = r[((r.TIPO == "cruce") & ~(r.DECISION.isin(DECISIONES_CRUCE) & r.FUENTE.isin({"ORBIS", "FDI"})))
              | ((r.TIPO == "duplicado") & ~(r.DECISION.isin(DECISIONES_DUP) & r.FUENTE.isin({"SEG", "ORBIS", "FDI"})))
              | ~r.TIPO.isin({"cruce", "duplicado"})]
    if len(malas):
        raise ValueError(f"Qué pasó: {len(malas)} filas de la revisión tienen TIPO, FUENTE o DECISION no válidos.\n"
                         f"Qué hacer: TIPO ∈ {{cruce, duplicado}}; FUENTE ∈ {{ORBIS, FDI}} en cruce y {{SEG, ORBIS, FDI}} en "
                         f"duplicado; DECISION ∈ {sorted(DECISIONES_CRUCE)} (cruce) o {sorted(DECISIONES_DUP)} (duplicado). "
                         f"Primeras:\n{malas.head().to_string()}")
    par = pd.Series([frozenset((a, b)) for a, b in zip(r.CLAVE_A, r.CLAVE_B)], index=r.index)
    dup = r[pd.DataFrame({"T": r.TIPO, "F": r.FUENTE, "P": par}).duplicated(keep=False)]
    if len(dup):
        raise ValueError("Qué pasó: la revisión decide dos veces el mismo par (en cualquier orden).\n"
                         f"Qué hacer: deje una sola fila por par. Primeros:\n{dup.head().to_string()}")
    return r


def validar_revision(revision: pd.DataFrame | None, prep: dict, claves: dict) -> None:
    """Detiene la corrida si la revisión no corresponde a estos archivos.

    Las claves de FDI son la fila del Excel (FDI no trae identificador): si la hoja se
    reordena o se vuelve a descargar, la misma clave apunta a otra empresa. Por eso cada
    fila de la revisión trae el NOMBRE que vio el revisor, y aquí se compara con el nombre
    actual de su clave. También detiene si una clave no existe.
    """
    if revision is None:
        return
    def norm(x):
        return _clave(pd.Series(x)).str.casefold().to_numpy()
    nombre = {k: dict(zip(claves[k], norm(prep[k].NOMBRE_ORIGINAL))) for k in prep}
    problemas = []
    for lado in ("A", "B"):
        fuente = np.where((revision.TIPO == "cruce") & (lado == "A"), "SEG", revision.FUENTE)
        for f, clave, nom in zip(fuente, revision[f"CLAVE_{lado}"], norm(revision[f"NOMBRE_{lado}"])):
            actual = nombre.get(f, {}).get(clave)
            if actual is None:
                problemas.append(f"{f} · {clave}: la clave no existe en los archivos")
            elif f != "SEG" and actual != nom:
                problemas.append(f"{f} · {clave}: la revisión dice {nom!r}, el archivo dice {actual!r}")
    if problemas:
        raise ValueError(f"Qué pasó: {len(problemas)} filas de la revisión no corresponden a estos archivos (primeras):\n  "
                         + "\n  ".join(problemas[:10]) + "\nQué hacer: use la revisión hecha sobre ESTOS archivos, o "
                         "ARCHIVO_REVISION = \"\" para correr solo el motor automático. En FDI la clave es la fila del Excel: "
                         "si la hoja cambió de orden, la revisión anterior ya no sirve.")


def enlaces_automaticos(pares: pd.DataFrame, seg: pd.DataFrame, dest: pd.DataFrame) -> pd.DataFrame:
    """Aceptados por el motor, una fila por (clave SEG, registro destino) con el mejor puntaje."""
    acc = pares[pares.CLASE != ""].copy()
    acc["CLAVE_A"] = clave_seg(seg)[acc.pos_seg.to_numpy()]
    acc["CLAVE_B"] = dest.ID_REGISTRO.to_numpy()[acc.pos_dest.to_numpy()]
    acc["_r"] = (acc.CLASE == "misma_empresa").astype(int)
    acc = acc.sort_values(["_r", "SCORE"], ascending=False).drop_duplicates(["CLAVE_A", "CLAVE_B"])
    return acc[["CLAVE_A", "CLAVE_B", "CLASE", "SCORE", "SIM_ESTRICTA", "SIM_GRUPO", "SOBRA_SEG", "SOBRA_DEST", "POR_ALIAS"]].reset_index(drop=True)


def aplicar_revision_cruce(auto: pd.DataFrame, revision: pd.DataFrame | None, fuente: str) -> pd.DataFrame:
    """Enlaces finales: la revisión confirma, cambia la clase, agrega o rechaza. Cada fila dice su ORIGEN."""
    a = auto.rename(columns={"CLASE": "CLASE_AUTO"})
    if revision is None:
        a["CLASE"] = a.CLASE_AUTO; a["ORIGEN"] = "automatico"; a["RAZON_REVISION"] = ""
        return a
    r = revision[(revision.TIPO == "cruce") & (revision.FUENTE == fuente)][["CLAVE_A", "CLAVE_B", "DECISION", "ORIGEN_REVISION", "RAZON"]]
    m = a.merge(r, on=["CLAVE_A", "CLAVE_B"], how="outer")
    m["CLASE"] = np.where(m.DECISION.notna() & (m.DECISION != ""), m.DECISION, m.CLASE_AUTO)
    m["ORIGEN"] = np.select(
        [m.DECISION.isna(), m.CLASE_AUTO.isna(), m.DECISION.isin(["distinta", "relacion_propiedad"]), m.DECISION == m.CLASE_AUTO],
        ["automatico", "revision_agrega", "revision_excluye", "revision_confirma"], default="revision_cambia_clase")
    m.loc[m.CLASE_AUTO.isna() & m.DECISION.isin(["distinta", "relacion_propiedad"]), "ORIGEN"] = "revision_sin_efecto"
    m["RAZON_REVISION"] = m.RAZON.fillna("")
    return m.drop(columns=["RAZON"])


def metricas_automatico(auto: pd.DataFrame, revision: pd.DataFrame | None, fuente: str) -> dict:
    """Precisión y recall del motor automático contra los enlaces adjudicados."""
    if revision is None:
        return {"fuente": fuente, "nota": "sin revisión: no hay contra qué medir"}
    ref = revision[(revision.TIPO == "cruce") & (revision.FUENTE == fuente) & revision.DECISION.isin(["misma_empresa", "mismo_grupo"])]
    A = set(zip(auto.CLAVE_A, auto.CLAVE_B)); R = set(zip(ref.CLAVE_A, ref.CLAVE_B))
    tp = len(A & R)
    out = {"fuente": fuente, "enlaces_automaticos": len(A), "enlaces_referencia": len(R), "aciertos": tp,
           "precision": round(tp / len(A), 4) if A else np.nan, "recall": round(tp / len(R), 4) if R else np.nan}
    out["f1"] = round(2 * out["precision"] * out["recall"] / (out["precision"] + out["recall"]), 4) if tp else 0.0
    for c in ("misma_empresa", "mismo_grupo"):
        Rc = set(zip(ref.loc[ref.DECISION == c, "CLAVE_A"], ref.loc[ref.DECISION == c, "CLAVE_B"]))
        Ac = set(zip(auto.loc[auto.CLASE == c, "CLAVE_A"], auto.loc[auto.CLASE == c, "CLAVE_B"]))
        out[f"recall_{c}"] = round(len(A & Rc) / len(Rc), 4) if Rc else np.nan
        out[f"precision_{c}"] = round(len(Ac & R) / len(Ac), 4) if Ac else np.nan
        out[f"clase_correcta_{c}"] = round(len(Ac & Rc) / len(Ac & R), 4) if (Ac & R) else np.nan
    return out


def agrupar(prep: pd.DataFrame, aristas: pd.DataFrame, revision: pd.DataFrame | None, fuente: str,
            clave: np.ndarray) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Entidades finales = uniones automáticas − las que la revisión veta + las que la revisión declara.

    La revisión decide por par de claves naturales: ``misma_entidad`` une; ``mismo_grupo`` o
    ``distinta`` veta la unión automática directa de ese par. Devuelve (prep con GRUPO y
    ORIGEN_GRUPO, tabla de vetos con su efecto) para que un veto que la transitividad
    anula (A~C y C~B siguen unidos) quede a la vista y lo detecte un invariante.
    """
    out = prep.copy()
    pos = pd.Series(np.arange(len(out))).groupby(np.asarray(clave)).first().to_dict()
    a = aristas.astype({"pos_a": "int64", "pos_b": "int64"})   # una concatenación con una tabla vacía las deja en object
    a["CA"], a["CB"] = np.asarray(clave)[a.pos_a.to_numpy()], np.asarray(clave)[a.pos_b.to_numpy()]
    vetos = pd.DataFrame(columns=["CLAVE_A", "CLAVE_B", "DECISION", "VETA_UNION_AUTOMATICA"])
    agregadas = pd.DataFrame({"pos_a": pd.Series(dtype="int64"), "pos_b": pd.Series(dtype="int64"), "REGLA": pd.Series(dtype=str)})
    if revision is not None:
        r = revision[(revision.TIPO == "duplicado") & (revision.FUENTE == fuente)]
        no = r[r.DECISION != "misma_entidad"]
        par_no = {frozenset(p) for p in zip(no.CLAVE_A, no.CLAVE_B)}
        veta = np.array([frozenset((x, y)) in par_no for x, y in zip(a.CA, a.CB)], dtype=bool)
        vetos = no[["CLAVE_A", "CLAVE_B", "DECISION"]].copy()
        vetados = {frozenset(p) for p in zip(a.CA[veta], a.CB[veta])}
        vetos["VETA_UNION_AUTOMATICA"] = [frozenset(p) in vetados for p in zip(vetos.CLAVE_A, vetos.CLAVE_B)]
        a = a[~veta]
        si = r[(r.DECISION == "misma_entidad") & r.CLAVE_A.isin(pos) & r.CLAVE_B.isin(pos)]
        agregadas = pd.DataFrame({"pos_a": si.CLAVE_A.map(pos).to_numpy(), "pos_b": si.CLAVE_B.map(pos).to_numpy(), "REGLA": "revisión"})
    todas = pd.concat([a[["pos_a", "pos_b", "REGLA"]], agregadas], ignore_index=True)
    comp = _componentes(len(out), todas.pos_a, todas.pos_b)
    out["GRUPO"] = [f"{fuente}-E{g:04d}" for g in comp]
    origen = todas.assign(G=comp[todas.pos_a.to_numpy().astype(int)] if len(todas) else []).groupby("G").REGLA.agg(lambda x: " + ".join(sorted(set(x))))
    out["ORIGEN_GRUPO"] = [origen.get(g, "") for g in comp]
    if len(vetos):
        g = dict(zip(np.asarray(clave), out.GRUPO))
        vetos["SIGUEN_JUNTOS"] = [g.get(x) is not None and g.get(x) == g.get(y) for x, y in zip(vetos.CLAVE_A, vetos.CLAVE_B)]
    else:
        vetos["SIGUEN_JUNTOS"] = pd.Series(dtype=bool)
    return out, vetos


def metricas_dedup(prep: pd.DataFrame, revision: pd.DataFrame | None, fuente: str, clave: np.ndarray) -> dict | None:
    """Deduplicación AUTOMÁTICA (GRUPO_DEDUP, antes de la revisión) contra los pares adjudicados.

    Solo cuenta pares que la referencia adjudicó: una unión automática fuera de esa lista no
    se puede calificar (se informa aparte, en N_UNIONES_NO_ADJUDICADAS).
    """
    if revision is None:
        return None
    r = revision[(revision.TIPO == "duplicado") & (revision.FUENTE == fuente)]
    g = dict(zip(np.asarray(clave), prep.GRUPO_DEDUP))
    R = {frozenset((a, b)): d for a, b, d in zip(r.CLAVE_A, r.CLAVE_B, r.DECISION) if a in g and b in g and a != b}
    unidos = {p for p in R if len({g[x] for x in p}) == 1}
    verdad = {p for p, d in R.items() if d == "misma_entidad"}
    tp = len(unidos & verdad)
    P = tp / len(unidos) if unidos else float("nan"); Rc = tp / len(verdad) if verdad else float("nan")
    # uniones automáticas entre claves distintas que nadie adjudicó
    todas = set()
    for _, x in pd.Series(np.asarray(clave)).groupby(prep.GRUPO_DEDUP.to_numpy()):
        u = sorted(set(x))
        todas |= {frozenset((u[0], v)) for v in u[1:]}
    return {"fuente": f"DEDUP_{fuente}", "enlaces_automaticos": len(unidos), "enlaces_referencia": len(verdad), "aciertos": tp,
            "precision": round(P, 4), "recall": round(Rc, 4), "f1": round(2 * P * Rc / (P + Rc), 4) if tp else 0.0,
            "N_UNIONES_NO_ADJUDICADAS": len({p for p in todas if p not in R})}


def aristas_por_destino(final: pd.DataFrame, clave: np.ndarray) -> pd.DataFrame:
    """Filas SEG que la decisión final enlaza como misma_empresa con el MISMO registro de ORBIS.

    Un BvD ID es una persona jurídica: si «G.E.P.S.A.U» y «GR.EMP. P. ALIMENTACION» son,
    cada una, la misma empresa que GRUPO EMPRESARIAL PALACIOS ALIMENTACION SAU, son la
    misma empresa entre sí. Solo ORBIS: un registro FDI puede ser una marca o un grupo.
    """
    pos = pd.Series(np.arange(len(clave))).groupby(np.asarray(clave)).first().to_dict()
    m = final[(final.CLASE == "misma_empresa") & final.CLAVE_A.isin(pos)]
    filas = []
    for dest, g in m.groupby("CLAVE_B"):
        ps = sorted({pos[k] for k in g.CLAVE_A})
        filas += [(ps[0], p, f"misma empresa que ORBIS {dest}") for p in ps[1:]]
    return pd.DataFrame(filas, columns=["pos_a", "pos_b", "REGLA"]).astype({"pos_a": "int64", "pos_b": "int64"})


def ultima_linea(cfg_dict: dict) -> None:
    print(json.dumps(cfg_dict, ensure_ascii=False, indent=1, default=str)[:2000])


def escribir_excel(tablas: dict[str, pd.DataFrame], ruta: Path) -> Path:
    """Excel multi-hoja con la ruta de exportación de la librería (neutraliza fórmulas y bytes de control)."""
    ruta.parent.mkdir(parents=True, exist_ok=True)
    vacias = {k: v for k, v in tablas.items() if v is None or len(v) == 0}
    llenas = {k: v for k, v in tablas.items() if k not in vacias}
    salida = Path(PipelineResult(work_dir=ruta.parent, extra=llenas).to_excel(ruta, include=tuple(llenas)))
    try:  # formato de lectura: encabezado fijo, filtro y anchos
        from openpyxl import load_workbook
        from openpyxl.styles import Alignment, Font, PatternFill
        wb = load_workbook(salida)
        for ws in wb.worksheets:
            ws.freeze_panes = "A2"
            if ws.max_row > 1:
                ws.auto_filter.ref = ws.dimensions
            for c in ws[1]:
                c.font = Font(bold=True, color="FFFFFF"); c.fill = PatternFill("solid", fgColor="1F4E78")
                c.alignment = Alignment(wrap_text=True, vertical="top")
            for col in ws.columns:
                largo = max((len(str(c.value)) if c.value is not None else 0) for c in list(col)[:300])
                ws.column_dimensions[col[0].column_letter].width = max(9, min(60, largo + 2))
        wb.save(salida)
    except Exception as exc:  # el formato es cosmético: nunca debe tumbar la exportación
        print(f"  ⚠️  sin formato de lectura ({exc}); el contenido está completo")
    return salida


# ── Tablas de resultado ───────────────────────────────────────────────────
_ORDEN_CLASE = {"misma_empresa": 0, "mismo_grupo": 1}


def _num(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce").fillna(0.0)


def _enlaces_por_entidad(final: pd.DataFrame, seg: pd.DataFrame) -> pd.DataFrame:
    """Enlaces finales (misma/grupo) llevados de la clave SEG a la entidad SEG deduplicada.

    La decisión humana manda sobre la ENTIDAD: si la revisión decidió el par (entidad, destino)
    para alguna de sus grafías, esa decisión vale para todas. Un enlace automático de otra
    grafía no revive un par que la revisión rechazó ni le cambia la clase que le puso.
    """
    grupo_de = dict(zip(clave_seg(seg), seg.GRUPO))
    todo = final.copy()
    todo["SEG_ID"] = todo.CLAVE_A.map(grupo_de)
    todo = todo[todo.SEG_ID.notna()]
    clase_rev = {}
    rev = todo[todo.DECISION.notna() & (todo.DECISION != "")] if "DECISION" in todo else todo.iloc[:0].assign(DECISION="")
    for k, d in rev.groupby(["SEG_ID", "CLAVE_B"]).DECISION:
        aceptadas = [x for x in set(d) if x in _ORDEN_CLASE]
        clase_rev[k] = min(aceptadas, key=_ORDEN_CLASE.get) if aceptadas else None
    f = todo[todo.CLASE.isin(list(_ORDEN_CLASE))].copy()
    f["_o"] = f.CLASE.map(_ORDEN_CLASE)
    f["SCORE_AUTO"] = pd.to_numeric(f.get("SCORE"), errors="coerce")
    f = f.sort_values(["SEG_ID", "_o", "SCORE_AUTO"], ascending=[True, True, False])
    agg = f.groupby(["SEG_ID", "CLAVE_B"], sort=False).agg(
        CLASE=("CLASE", "first"), SCORE_AUTO=("SCORE_AUTO", "max"),
        ORIGEN=("ORIGEN", lambda s: " + ".join(sorted(set(s)))),
        RAZON_REVISION=("RAZON_REVISION", lambda s: next((x for x in s if x), "")),
    ).reset_index()
    if clase_rev:
        k = list(zip(agg.SEG_ID, agg.CLAVE_B))
        agg["CLASE"] = [clase_rev[x] if x in clase_rev else c for x, c in zip(k, agg.CLASE)]
        agg = agg[agg.CLASE.notna()].copy()
    # Evidencia del motor: solo de filas que el motor aceptó (las que agrega la revisión no la tienen)
    auto = f[f.CLASE_AUTO.notna()] if "CLASE_AUTO" in f else f
    ev = {(s_, b): f"sobra en SEG: [{a or ''}] · sobra en destino: [{d or ''}]" + (" · por alias" if al is True else "")
          for s_, b, a, d, al in list(zip(auto.SEG_ID, auto.CLAVE_B, auto.SOBRA_SEG, auto.SOBRA_DEST, auto.POR_ALIAS))[::-1]}
    agg["EVIDENCIA_AUTO"] = [ev.get((s_, b), "") for s_, b in zip(agg.SEG_ID, agg.CLAVE_B)]
    agg["_o"] = agg.CLASE.map(_ORDEN_CLASE)
    agg = agg.sort_values(["SEG_ID", "_o", "SCORE_AUTO"], ascending=[True, True, False])
    agg["ES_MEJOR"] = ~agg.duplicated("SEG_ID")
    return agg.drop(columns="_o")


def _entidades_seg(crudo: pd.DataFrame, seg: pd.DataFrame, f, sin_contraparte: list[str]) -> pd.DataFrame:
    s = seg.copy()
    for c in f.cols_metricas:
        s[c] = _num(crudo[c]).to_numpy()
    s["HS6"] = crudo["HS6"].to_numpy() if "HS6" in crudo else ""
    agr = {"EMPRESA": ("NOMBRE_ORIGINAL", lambda x: x.value_counts().index[0]),
           "GRAFIAS": ("NOMBRE_ORIGINAL", lambda x: " | ".join(sorted(set(x)))),
           "PAIS": ("PAIS_ORIGINAL", "first"), "PAIS_ISO3": ("PAIS_ISO3", "first"), "N_FILAS": ("NOMBRE_ORIGINAL", "size"),
           **{c: (c, "sum") for c in f.cols_metricas},
           "HS6": ("HS6", lambda x: ", ".join(sorted(set(map(str, x))))),
           "TRUNCADO_35": ("TRUNCADO", "any"), "INTERMEDIARIO_LOGISTICO": ("INTERMEDIARIO", "any")}
    e = s.groupby("GRUPO").agg(**agr).reset_index().rename(columns={"GRUPO": "SEG_ID"})
    e["PAIS_SIN_CONTRAPARTE"] = e.PAIS_ISO3.isin(sin_contraparte)
    return e


def _numerico_si_puede(s: pd.Series) -> pd.Series:
    """Las fuentes se leen como texto (para no perder ceros ni IDs); una columna que es toda numérica vuelve a número.

    Los marcadores de dato ausente de Orbis ('n.a.', 'n.s.') cuentan como vacío.
    """
    txt = s.astype(str).str.strip().str.lower()
    llenas = s.notna() & (txt != "") & ~txt.isin({"n.a.", "n.s.", "n/a", "na", "-", "—"})
    t = pd.to_numeric(s.where(llenas), errors="coerce")
    return t if llenas.any() and t[llenas].notna().all() else s


def _contexto(prep: pd.DataFrame, crudo: pd.DataFrame, f, prefijo: str) -> pd.DataFrame:
    c = pd.DataFrame({"CLAVE_B": prep.ID_REGISTRO.to_numpy(), f"{prefijo}_NOMBRE": prep.NOMBRE_ORIGINAL.to_numpy(),
                      f"{prefijo}_PAIS": prep.PAIS_ORIGINAL.to_numpy(), f"{prefijo}_FILA_EXCEL": crudo.FILA_EXCEL.to_numpy(),
                      f"{prefijo}_ENTIDAD": prep.GRUPO.to_numpy(), "_ISO_DEST": prep.PAIS_ISO3.to_numpy()})
    for col in f.cols_contexto:
        c[f"{prefijo}_{col}"] = _numerico_si_puede(crudo[col]).to_numpy()
    return c


def construir_tablas(crudo, prep, final, pares, revision, medicion, fuentes, cfg, sin_contraparte, dedup_pend, dedup_vetos):
    """Todas las hojas del Excel y la tabla de invariantes."""
    fs = fuentes["SEG"]
    ent = _entidades_seg(crudo["SEG"], prep["SEG"], fs, sin_contraparte)
    match, mejores = {}, {}
    for d, pref in (("ORBIS", "ORBIS"), ("FDI", "FDI")):
        e = _enlaces_por_entidad(final[d], prep["SEG"])
        ctx = _contexto(prep[d], crudo[d], fuentes[d], pref)
        m = e.merge(ctx, on="CLAVE_B", how="left").merge(ent[["SEG_ID", "EMPRESA", "GRAFIAS", "PAIS", "PAIS_ISO3"]], on="SEG_ID", how="left")
        m = m.rename(columns={"CLAVE_B": f"{pref}_ID", "EMPRESA": "SEG_EMPRESA", "GRAFIAS": "SEG_GRAFIAS", "PAIS": "SEG_PAIS"})
        m["_ISO_OK"] = m._ISO_DEST == m.PAIS_ISO3
        cols = ["SEG_ID", "SEG_EMPRESA", "SEG_GRAFIAS", "SEG_PAIS", "PAIS_ISO3", "CLASE", "ES_MEJOR", "ORIGEN", "SCORE_AUTO",
                f"{pref}_ID", f"{pref}_NOMBRE", f"{pref}_PAIS", f"{pref}_FILA_EXCEL", f"{pref}_ENTIDAD",
                *[f"{pref}_{c}" for c in fuentes[d].cols_contexto], "EVIDENCIA_AUTO", "RAZON_REVISION", "_ISO_OK", "_ISO_DEST"]
        match[d] = m[cols]
        b = m[m.ES_MEJOR].set_index("SEG_ID")
        n = m.groupby("SEG_ID").size()
        mejores[d] = pd.DataFrame({f"{pref}_MEJOR_ID": b[f"{pref}_ID"], f"{pref}_MEJOR_NOMBRE": b[f"{pref}_NOMBRE"],
                                   f"{pref}_MEJOR_CLASE": b["CLASE"], f"N_ENLACES_{pref}": n})
    cons = ent.set_index("SEG_ID").join(mejores["ORBIS"]).join(mejores["FDI"]).reset_index()
    for pref in ("ORBIS", "FDI"):
        cons[f"N_ENLACES_{pref}"] = cons[f"N_ENLACES_{pref}"].fillna(0).astype(int)
    tiene_o, tiene_f = cons.N_ENLACES_ORBIS > 0, cons.N_ENLACES_FDI > 0
    cons["ESTADO"] = np.select([cons.PAIS_SIN_CONTRAPARTE, tiene_o & tiene_f, tiene_o, tiene_f],
                               ["país sin contraparte", "ORBIS y FDI", "solo ORBIS", "solo FDI"], default="sin coincidencia")
    # Correlativa fila a fila (el insumo, tal cual, con su entidad y su mejor enlace)
    filas = crudo["SEG"].drop(columns="FILA_EXCEL").copy()
    for col in fs.cols_metricas:
        filas[col] = _numerico_si_puede(filas[col])
    filas.insert(0, "FILA_EXCEL", crudo["SEG"].FILA_EXCEL.to_numpy())
    filas.insert(1, "SEG_ID", prep["SEG"].GRUPO.to_numpy())
    filas = filas.merge(cons[["SEG_ID", "ESTADO", "ORBIS_MEJOR_ID", "ORBIS_MEJOR_NOMBRE", "ORBIS_MEJOR_CLASE",
                              "FDI_MEJOR_ID", "FDI_MEJOR_NOMBRE", "FDI_MEJOR_CLASE"]], on="SEG_ID", how="left")
    # Duplicados internos (grupos con más de un registro)
    dedup = {}
    for k in ("SEG", "ORBIS", "FDI"):
        p = prep[k]
        if k == "SEG":
            # SEG repite la misma grafía en varias filas (una por producto): eso no es un duplicado.
            # Se listan las entidades con MÁS DE UNA grafía, una fila por grafía con su número de filas.
            g = p.assign(CLAVE=clave_seg(p)).groupby(["GRUPO", "CLAVE"], sort=False).agg(
                NOMBRE=("NOMBRE_ORIGINAL", "first"), PAIS=("PAIS_ORIGINAL", "first"), NOMBRE_NORMALIZADO=("NOMBRE_NORM", "first"),
                FORMA_LEGAL=("FORMA_LEGAL", "first"), N_FILAS=("NOMBRE_ORIGINAL", "size"), UNIDO_POR=("ORIGEN_GRUPO", "first")).reset_index()
            g = g[g.groupby("GRUPO").CLAVE.transform("size") > 1]
            dedup[k] = g.rename(columns={"GRUPO": "ENTIDAD", "CLAVE": "ID_REGISTRO"})[
                ["ENTIDAD", "ID_REGISTRO", "NOMBRE", "PAIS", "NOMBRE_NORMALIZADO", "FORMA_LEGAL", "N_FILAS", "UNIDO_POR"]].sort_values(["ENTIDAD", "NOMBRE"])
            continue
        g = p[p.groupby("GRUPO").GRUPO.transform("size") > 1]
        dedup[k] = pd.DataFrame({"ENTIDAD": g.GRUPO, "ID_REGISTRO": g.ID_REGISTRO,
                                 "NOMBRE": g.NOMBRE_ORIGINAL, "PAIS": g.PAIS_ORIGINAL, "NOMBRE_NORMALIZADO": g.NOMBRE_NORM,
                                 "FORMA_LEGAL": g.FORMA_LEGAL, "UNIDO_POR": g.ORIGEN_GRUPO}).sort_values(["ENTIDAD", "NOMBRE"])
    # Pares de duplicado que el motor no une y nadie revisó (homónimos, siglas, forma legal distinta)
    revisados_dup = set() if revision is None else {frozenset(p) for p in zip(revision.CLAVE_A, revision.CLAVE_B)}
    dpend = []
    for k in ("SEG", "ORBIS", "FDI"):
        q, p = dedup_pend[k], prep[k]
        if not len(q):
            continue
        ck = clave_seg(p) if k == "SEG" else p.ID_REGISTRO.to_numpy()
        ca, cb = ck[q.pos_a.to_numpy()], ck[q.pos_b.to_numpy()]
        mismo = p.GRUPO.to_numpy()[q.pos_a.to_numpy()] == p.GRUPO.to_numpy()[q.pos_b.to_numpy()]
        keep = np.array([frozenset((x, y)) not in revisados_dup for x, y in zip(ca, cb)]) & ~mismo
        dpend.append(pd.DataFrame({"FUENTE": k, "CLAVE_A": ca, "NOMBRE_A": p.NOMBRE_ORIGINAL.to_numpy()[q.pos_a.to_numpy()],
                                   "CLAVE_B": cb, "NOMBRE_B": p.NOMBRE_ORIGINAL.to_numpy()[q.pos_b.to_numpy()],
                                   "PAIS": p.PAIS_ORIGINAL.to_numpy()[q.pos_a.to_numpy()], "MOTIVO": q.MOTIVO.to_numpy()})[keep])
    dpend = pd.concat(dpend, ignore_index=True) if dpend else pd.DataFrame(columns=["FUENTE", "CLAVE_A", "NOMBRE_A", "CLAVE_B", "NOMBRE_B", "PAIS", "MOTIVO"])
    if len(dpend):
        par = pd.Series([frozenset((a, b)) for a, b in zip(dpend.CLAVE_A, dpend.CLAVE_B)], index=dpend.index)
        dpend = dpend[~pd.DataFrame({"F": dpend.FUENTE, "P": par}).duplicated().to_numpy()
                      & (dpend.CLAVE_A != dpend.CLAVE_B).to_numpy()].reset_index(drop=True)
    # Revisión: relaciones de propiedad, exclusiones y pendientes
    prop = excl = pd.DataFrame()
    if revision is not None:
        prop = revision[(revision.TIPO == "cruce") & (revision.DECISION == "relacion_propiedad")].copy()
        prop.insert(0, "SEG_ID", prop.CLAVE_A.map(dict(zip(clave_seg(prep["SEG"]), prep["SEG"].GRUPO))))
        prop = prop[["SEG_ID", "FUENTE", "NOMBRE_A", "CLAVE_B", "NOMBRE_B", "RAZON"]].sort_values(["SEG_ID", "FUENTE", "NOMBRE_B"])
        excl = pd.concat([final[d].assign(FUENTE=d) for d in final])
        excl = excl[excl.ORIGEN == "revision_excluye"][["FUENTE", "CLAVE_A", "CLAVE_B", "CLASE_AUTO", "SCORE", "DECISION", "RAZON_REVISION"]]
    pend = []
    revisados = set() if revision is None else set(zip(revision.CLAVE_A, revision.CLAVE_B))
    for d in ("ORBIS", "FDI"):
        p = pares[d]
        cerca = p[(p.CLASE == "") & ((p.SIM_ESTRICTA >= 0.75) | (p.SIM_NUCLEO >= 0.84) | p.COMPARTE_DISTINTIVA | p.ABREVIATURA)].copy()
        cerca["CLAVE_A"] = clave_seg(prep["SEG"])[cerca.pos_seg.to_numpy()]
        cerca["CLAVE_B"] = prep[d].ID_REGISTRO.to_numpy()[cerca.pos_dest.to_numpy()]
        cerca = cerca[np.array([(a, b) not in revisados for a, b in zip(cerca.CLAVE_A, cerca.CLAVE_B)], dtype=bool)].drop_duplicates(["CLAVE_A", "CLAVE_B"])
        cerca["NOMBRE_DESTINO"] = prep[d].NOMBRE_ORIGINAL.to_numpy()[cerca.pos_dest.to_numpy()]
        pend.append(cerca.assign(FUENTE=d)[["FUENTE", "CLAVE_A", "CLAVE_B", "NOMBRE_DESTINO", "SIM_ESTRICTA", "SIM_NUCLEO", "COMPARTE_DISTINTIVA", "ABREVIATURA", "SOBRA_SEG", "SOBRA_DEST"]])
    pend = pd.concat(pend, ignore_index=True).round(3) if pend else pd.DataFrame()

    # ── Invariantes ──
    total_in = {c: _num(crudo["SEG"][c]).sum() for c in fs.cols_metricas}
    inv = []
    def chk(nombre, cumple, detalle=""):
        inv.append({"invariante": nombre, "cumple": bool(cumple), "detalle": detalle})
    chk("SEG_FILAS tiene una fila por fila de entrada", len(filas) == len(crudo["SEG"]), f"{len(filas)} de {len(crudo['SEG'])}")
    chk("cada fila SEG pertenece a exactamente una entidad", prep["SEG"].GRUPO.notna().all() and filas.SEG_ID.notna().all())
    for c in fs.cols_metricas:
        chk(f"{c} se conserva al consolidar", abs(cons[c].sum() - total_in[c]) <= 1e-6 * max(1.0, abs(total_in[c])), f"{cons[c].sum():,.2f}")
    for d in ("ORBIS", "FDI"):
        m = match[d]
        chk(f"ningún enlace SEG↔{d} cruza países", bool(m._ISO_OK.all()), f"{int((~m._ISO_OK).sum())} fuera de país")
        chk(f"todo ID enlazado existe en {d}", m[f"{d}_NOMBRE"].notna().all())
        chk(f"sin pares repetidos en MATCH_SEG_{d}", not m.duplicated(["SEG_ID", f"{d}_ID"]).any())
        chk(f"clases válidas en MATCH_SEG_{d}", m.CLASE.isin(list(_ORDEN_CLASE)).all())
        pm = pares[d][pares[d].CLASE == "misma_empresa"]
        mal = pm[(pm.SIM_ESTRICTA < cfg.umbral_misma_empresa - 1e-9) | (pm.SOBRA_SEG != "") | (pm.SOBRA_DEST != "")]
        chk(f"toda 'misma_empresa' automática SEG↔{d} tiene todas las palabras alineadas y supera el umbral", mal.empty,
            f"{len(mal)} no cumplen")
        if revision is not None:
            r = revision[(revision.TIPO == "cruce") & (revision.FUENTE == d)]
            g = dict(zip(clave_seg(prep["SEG"]), prep["SEG"].GRUPO))
            dec = pd.DataFrame({"SEG_ID": r.CLAVE_A.map(g), "B": r.CLAVE_B, "D": r.DECISION}).dropna()
            rechazados = {k for k, x in dec.groupby(["SEG_ID", "B"]).D if not set(x) & set(_ORDEN_CLASE)}
            vivos = [k for k in zip(m.SEG_ID, m[f"{d}_ID"]) if k in rechazados]
            chk(f"ningún par SEG↔{d} que la revisión rechazó vuelve a entrar por otra grafía", not vivos, f"{len(vivos)} reingresan")
        chk(f"países sin contraparte no tienen enlaces con {d}", not m.PAIS_ISO3.isin(sin_contraparte).any())
    for k in ("SEG", "ORBIS", "FDI"):
        p = prep[k]
        chk(f"cada entidad {k} tiene un solo país", (p.groupby("GRUPO").PAIS_ISO3.nunique() <= 1).all())
        v = dedup_vetos[k]
        juntos = v[v.SIGUEN_JUNTOS] if len(v) else v
        chk(f"ningún par que la revisión separa en {k} queda en la misma entidad", juntos.empty,
            f"{len(juntos)} siguen juntos por otra vía" + (f", p. ej. {list(zip(juntos.CLAVE_A, juntos.CLAVE_B))[:2]}" if len(juntos) else ""))
    if revision is not None:
        claves = set(clave_seg(prep["SEG"])) | set(prep["ORBIS"].ID_REGISTRO) | set(prep["FDI"].ID_REGISTRO)
        desconocidas = [k for k in pd.concat([revision.CLAVE_A, revision.CLAVE_B]).unique() if k not in claves]
        chk("toda clave de la revisión existe en las bases (la revisión corresponde a estos archivos)", not desconocidas,
            f"{len(desconocidas)} desconocidas" + (f", p. ej. {desconocidas[:3]}" if desconocidas else ""))
    INV = pd.DataFrame(inv)
    for d in ("ORBIS", "FDI"):
        match[d] = match[d].drop(columns=["_ISO_OK", "_ISO_DEST"])

    # ── Resumen ──
    n_ent = len(cons); n_cp = int((~cons.PAIS_SIN_CONTRAPARTE).sum())
    # métricas con algún valor (en el piloto, Valor U$S viene todo en cero)
    vals = [c for c in fs.cols_metricas if cons[c].abs().sum() > 0]
    def pct(x, tot): return f"{x:,} ({x / tot:.1%})" if tot else str(x)
    res = [("filas SEG de entrada", f"{len(crudo['SEG']):,}"), ("empresas SEG tras deduplicar", f"{n_ent:,}"),
           ("empresas SEG en países sin contraparte", f"{n_ent - n_cp:,} ({', '.join(sin_contraparte) or '—'})"),
           ("registros ORBIS / FDI", f"{len(crudo['ORBIS']):,} / {len(crudo['FDI']):,}")]
    for d in ("ORBIS", "FDI"):
        m = match[d]
        con = m.SEG_ID.nunique(); misma = m[m.CLASE == "misma_empresa"].SEG_ID.nunique()
        res += [(f"empresas SEG con enlace a {d}", pct(con, n_cp)), (f"  · con al menos una 'misma_empresa'", pct(misma, n_cp)),
                (f"  · solo 'mismo_grupo'", pct(con - misma, n_cp)), (f"enlaces SEG↔{d} (todas las clases)", f"{len(m):,}")]
        for val in vals:
            v = cons.loc[cons[f"N_ENLACES_{d}"] > 0, val].sum(); tot = cons[val].sum()
            res.append((f"  · {val} de las empresas con enlace a {d}", f"{v:,.0f} de {tot:,.0f} ({v / tot:.1%})"))
    alguno = cons[(cons.N_ENLACES_ORBIS > 0) | (cons.N_ENLACES_FDI > 0)]
    res += [("empresas SEG con enlace en ORBIS o FDI", pct(len(alguno), n_cp))]
    for val in vals:
        res.append((f"  · {val} de las empresas con algún enlace", f"{alguno[val].sum():,.0f} de {cons[val].sum():,.0f} ({alguno[val].sum() / cons[val].sum():.1%})"))
    res += [("empresas SEG con enlace en ORBIS y FDI", pct(int((cons.ESTADO == "ORBIS y FDI").sum()), n_cp)),
            ("empresas SEG sin coincidencia (con país contraparte)", pct(int((cons.ESTADO == "sin coincidencia").sum()), n_cp)),
            ("fuente de las decisiones", "automático + revisión adjudicada" if revision is not None else "solo automático"),
            ("invariantes", f"{int(INV.cumple.sum())} de {len(INV)} cumplen")]
    RES = pd.DataFrame(res, columns=["indicador", "valor"])

    tablas = {"LEEME": leeme(revision is not None), "RESUMEN": RES, "SEG_CONSOLIDADO": cons,
              "MATCH_SEG_ORBIS": match["ORBIS"], "MATCH_SEG_FDI": match["FDI"], "SEG_FILAS": filas,
              "DEDUP_SEG": dedup["SEG"], "DEDUP_ORBIS": dedup["ORBIS"], "DEDUP_FDI": dedup["FDI"],
              "RELACIONES_PROPIEDAD": prop, "EXCLUIDOS_REVISION": excl, "REVISION_PENDIENTE": pend, "DEDUP_PENDIENTE": dpend,
              "VERIFICACION": medicion, "INVARIANTES": INV}
    return tablas, INV


def leeme(con_revision: bool) -> pd.DataFrame:
    filas = [
        ("Qué es", "Vinculación de la base de segmentación (Empresa + País de Empresa) con ORBIS (Company name Latin alphabet + Country) y FDI (Company name + País de consulta), y deduplicación interna de las tres bases. Motor: rues-linker."),
        ("Regla de país", "Solo se enlazan registros del MISMO país. El país se canoniza contra el catálogo ISO 3166 de la librería (España = Spain, Alemania = Germany). Países Bajos no existe en ORBIS ni en FDI: sus empresas no pueden tener coincidencia."),
        ("Clase misma_empresa", "El registro destino es la misma empresa: iguales salvo forma legal, truncamiento, puntuación, tildes o erratas evidentes."),
        ("Clase mismo_grupo", "Entidades distintas del mismo grupo que comparten un nombre distintivo (p. ej. LIDL EXPORT INTERNATIONAL ↔ LIDL POLSKA; EDEKA FOODSERVICE ↔ Edeka). Úselas para contexto de grupo, no como la misma empresa."),
        ("Relaciones de propiedad", "Vínculos por dueño último o matriz SIN nombre compartido (p. ej. Picnic GmbH, cuyo dueño es EDEKA). No son coincidencias de nombre: van aparte, en RELACIONES_PROPIEDAD."),
        ("FDI · País de consulta", "Es el país del inversionista consultado en fDi Markets, no la sede de cada filial: 'Lidl Polska' aparece bajo Alemania. Por eso las filiales extranjeras de un grupo alemán enlazan con empresas SEG alemanas."),
        ("SEG_CONSOLIDADO", "Una fila por empresa SEG deduplicada, con sus métricas sumadas, su mejor enlace en ORBIS y en FDI, y el ESTADO."),
        ("MATCH_SEG_ORBIS / MATCH_SEG_FDI", "Todos los enlaces (uno por par), con clase, ORIGEN, puntaje del motor y el contexto del registro destino. ES_MEJOR marca el enlace principal de cada empresa."),
        ("SEG_FILAS", "El insumo fila por fila con su SEG_ID y su mejor enlace: para cruzar de vuelta con cualquier otra tabla."),
        ("ORIGEN", "automatico = lo decidió el motor · revision_confirma = el motor y la revisión coinciden · revision_agrega = lo encontró la revisión y el motor no · revision_cambia_clase = coinciden en el enlace pero no en la clase."),
        ("VERIFICACION", "Precisión y recall del motor automático (sin la revisión) contra la referencia adjudicada: el cruce SEG↔ORBIS y SEG↔FDI, y la deduplicación de cada base (filas DEDUP_*; solo se califican pares adjudicados, las demás uniones se cuentan en N_UNIONES_NO_ADJUDICADAS)." if con_revision else "Sin referencia adjudicada: no se midió precisión ni recall."),
        ("REVISION_PENDIENTE", "Pares SEG↔destino que el motor no aceptó y nadie revisó: nombres cercanos, marca distintiva compartida (EDEKA MINDEN) o sigla/abreviatura (GR.EMP. P. ALIMENTACION). Es la cola de revisión humana: el motor no decide estos casos."),
        ("DEDUP_SEG / DEDUP_ORBIS / DEDUP_FDI", "Registros que son la MISMA entidad dentro de cada base. ORBIS: por identificador (BvD raíz; en Francia, SIREN). SEG: mismo nombre (erratas de una letra) con la misma forma legal, o misma empresa que un mismo registro ORBIS; en DEDUP_SEG solo aparecen las empresas con más de una grafía. FDI: solo se unen filas con el nombre idéntico; el resto lo decide la revisión (dos nombres parecidos en fDi suelen ser matriz y filial). UNIDO_POR dice qué regla o si fue la revisión. Filiales de un grupo NO se unen: son entidades distintas."),
        ("DEDUP_PENDIENTE", "Pares dentro de una misma base que el motor no une y nadie revisó: homónimos con identificador distinto, mismo nombre con forma legal distinta, coincidencia solo por el paréntesis, siglas."),
        ("Referencia adjudicada", "Construida por tres revisores independientes por lote (métodos distintos) y tres árbitros escépticos para cada desacuerdo. No es una verdad de campo: su error residual no está medido."),
    ]
    return pd.DataFrame(filas, columns=["tema", "detalle"])


def parametros(cfg, fuentes, huellas, archivo_revision, segundos) -> pd.DataFrame:
    import platform
    import record_linkage
    filas = [("librería", f"rues-linker {record_linkage.__version__}"), ("python", platform.python_version()),
             ("fecha", datetime.now().isoformat(timespec="seconds")), ("segundos", f"{segundos:.1f}"),
             ("archivo_revision", archivo_revision or "(ninguno)")]
    filas += [(f"sha256 · {Path(a).name}", h) for a, h in huellas.items()]
    for k, f in fuentes.items():
        filas.append((f"fuente {k}", f"hoja={f.hoja!r} nombre={f.col_nombre!r} país={f.col_pais!r} id={f.col_id!r} truncamiento={f.largo_truncamiento}"))
    for k, v in vars(cfg).items():
        filas.append((k, ", ".join(sorted(map(str, v))) if isinstance(v, (set, frozenset)) else str(v)))
    return pd.DataFrame(filas, columns=["parametro", "valor"])

## 4 · Lectura
Lee las tres hojas como texto, verifica que existan las columnas declaradas y guarda la huella SHA-256 de cada archivo.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  4 · LECTURA Y CONTRATO DE ENTRADA
# ══════════════════════════════════════════════════════════════════════════
import re
import time
from datetime import datetime

import numpy as np
import pandas as pd

T0 = time.time()
FUENTES_SPEC = {k: Fuente(clave=k, **v) for k, v in FUENTES.items()}
CFG = ConfigVinculacion(
    alias_paises=ALIAS_PAISES, formas_legales_extra=FORMAS_LEGALES_EXTRA, genericos_extra=GENERICOS_EXTRA,
    patrones_intermediario=PATRONES_INTERMEDIARIO, umbral_misma_empresa=UMBRAL_MISMA_EMPRESA,
    max_ediciones_palabra=MAX_EDICIONES_PALABRA,
)
CRUDO = {k: leer_fuente(f) for k, f in FUENTES_SPEC.items()}
HUELLAS = {f.archivo: sha256_archivo(f.archivo) for f in FUENTES_SPEC.values()}
for k, df in CRUDO.items():
    f = FUENTES_SPEC[k]
    print(f"✅ {k:<5} {len(df):>6,} filas · hoja {f.hoja!r} · nombre={f.col_nombre!r} · país={f.col_pais!r}"
          + (f" · id={f.col_id!r}" if f.col_id else ""))
for c in FUENTES_SPEC["SEG"].cols_metricas:
    v = pd.to_numeric(CRUDO["SEG"][c], errors="coerce")
    malos = v.isna() & CRUDO["SEG"][c].notna() & (CRUDO["SEG"][c].astype(str).str.strip() != "")
    print(f"   SEG · {c:<12} suma {v.sum():>18,.2f} · no numéricos {int(malos.sum())}")
    if malos.any():
        ej = CRUDO["SEG"].loc[malos, ["FILA_EXCEL", c]].head(8).to_string(index=False)
        raise ValueError(f"Qué pasó: '{c}' trae {int(malos.sum())} valores que no son números; se sumarían como 0.\n{ej}\n"
                         "Qué hacer: corríjalos en el Excel (sin separador de miles ni unidades) y vuelva a correr.")

## 5 · Preparación
País canónico y nombre normalizado con la librería. Si aparece una grafía de país que el catálogo no reconoce, **se detiene aquí** y la lista: agréguela en la celda 2.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  5 · PREPARACIÓN: PAÍS CANÓNICO Y NOMBRE NORMALIZADO (librería)
# ══════════════════════════════════════════════════════════════════════════
INTER = re.compile("|".join(CFG.patrones_intermediario), re.I) if CFG.patrones_intermediario else None
PREP = {k: preparar_fuente(CRUDO[k], FUENTES_SPEC[k], CFG, INTER) for k in FUENTES_SPEC}
for k, p in PREP.items():
    print(f"{k:<5} países: {p.PAIS_ISO3.value_counts().to_dict()}")
    print(f"      nombres: {p.NOMBRE_ORIGINAL.nunique():,} grafías → {p.NOMBRE_NORM.nunique():,} normalizadas · "
          f"sin nombre {int((p.NOMBRE_NORM == '').sum())} · con alias {int((p.ALIAS_NORM != '').sum())}"
          + (f" · truncados {int(p.TRUNCADO.sum())} · intermediarios {int(p.INTERMEDIARIO.sum())}" if k == "SEG" else ""))
for k, p in PREP.items():
    vacios = p.NOMBRE_ORIGINAL.str.strip() == ""
    if vacios.any():
        print(f"⚠️  {k}: {int(vacios.sum())} filas sin nombre (filas Excel {CRUDO[k].loc[vacios.to_numpy(), 'FILA_EXCEL'].tolist()[:15]}):"
              " cada una queda como una entidad aparte y no se enlaza.")
paises_destino = set(PREP["ORBIS"].PAIS_ISO3) | set(PREP["FDI"].PAIS_ISO3)
SIN_CONTRAPARTE = sorted(set(PREP["SEG"].PAIS_ISO3) - paises_destino)
if SIN_CONTRAPARTE:
    n = int(PREP["SEG"].PAIS_ISO3.isin(SIN_CONTRAPARTE).sum())
    print(f"\n⚠️  {n} filas SEG son de países que no están en ORBIS ni en FDI: {SIN_CONTRAPARTE}."
          "\n   Con el país como regla de cruce, no pueden tener coincidencia. No es un error del motor.")
print("\n🔎 intermediarios logísticos en SEG (solo se enlazan como 'misma_empresa'):")
print("   " + " · ".join(sorted(PREP["SEG"].loc[PREP["SEG"].INTERMEDIARIO, "NOMBRE_ORIGINAL"].unique())))

## 6 · Deduplicación interna
ORBIS por identificador; SEG por nombre estricto con la misma forma legal; FDI solo nombres idénticos (el resto, con revisión). La revisión puede **unir** registros y **vetar** uniones automáticas; si un veto queda anulado por otra vía, lo detecta una invariante.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  6 · DEDUPLICACIÓN INTERNA DE CADA BASE
# ══════════════════════════════════════════════════════════════════════════
#  ORBIS: por identificador (BvD raíz / SIREN). SEG y FDI: regla estricta de nombre con
#  forma legal compatible. La revisión (si la hay) une lo que falta y veta uniones erróneas.
t = time.time()
REVISION = leer_revision(ARCHIVO_REVISION)
HUELLA_REVISION = sha256_archivo(ARCHIVO_REVISION) if REVISION is not None else None
CLAVE = {"SEG": clave_seg(PREP["SEG"]), "ORBIS": PREP["ORBIS"].ID_REGISTRO.to_numpy(), "FDI": PREP["FDI"].ID_REGISTRO.to_numpy()}
validar_revision(REVISION, PREP, CLAVE)   # detiene si la revisión es de otros archivos


def revision_vigente() -> None:
    """Si el CSV de revisión cambió después de la celda 6, lo que sigue usaría la versión vieja."""
    if REVISION is not None and sha256_archivo(ARCHIVO_REVISION) != HUELLA_REVISION:
        raise RuntimeError("Qué pasó: el archivo de revisión cambió después de la celda 6.\n"
                           "Qué hacer: vuelva a correr desde la celda 6 (Deduplicación).")


CORPUS = pd.concat([p[c] for p in PREP.values() for c in ("NOMBRE_NORM", "ALIAS_NORM")])
COMPS = construir_comparadores(CORPUS, CFG)
ARISTAS, DEDUP_PEND, DEDUP_VETOS = {}, {}, {}
for k in PREP:
    PREP[k], ARISTAS[k], DEDUP_PEND[k] = deduplicar(PREP[k], FUENTES_SPEC[k], CFG, COMPS)
    PREP[k], DEDUP_VETOS[k] = agrupar(PREP[k], ARISTAS[k], REVISION, k, CLAVE[k])
    g = PREP[k].groupby("GRUPO").size()
    print(f"{k:<5} {len(PREP[k]):>5} filas → {len(g):>5} entidades · grupos con >1 registro: {int((g > 1).sum()):>3}"
          f" · automático: {PREP[k].GRUPO_DEDUP.nunique()} entidades"
          + (f" · la revisión veta {int(DEDUP_VETOS[k].VETA_UNION_AUTOMATICA.sum())} uniones" if len(DEDUP_VETOS[k]) else "")
          + f" · pendientes de mirar: {len(DEDUP_PEND[k])}")
print(f"⏱️  {time.time() - t:.1f} s")

## 7 · Vinculación
Comparación exhaustiva dentro de cada país y clasificación de cada par.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  7 · VINCULACIÓN SEG ↔ ORBIS y SEG ↔ FDI (exhaustiva dentro de cada país)
# ══════════════════════════════════════════════════════════════════════════
t = time.time()
PARES, AUTO = {}, {}
for d in ("ORBIS", "FDI"):
    PARES[d] = vincular(PREP["SEG"], PREP[d], COMPS, CFG)
    AUTO[d] = enlaces_automaticos(PARES[d], PREP["SEG"], PREP[d])
    print(f"SEG × {d:<5}: {len(PARES[d]):>8,} pares evaluados · {len(AUTO[d]):>4} enlaces automáticos "
          f"{AUTO[d].CLASE.value_counts().to_dict()}")
print(f"⏱️  {time.time() - t:.1f} s")

## 8 · Revisión adjudicada y medición
Aplica las decisiones del archivo de revisión (si existe) y mide al motor automático contra ellas: precisión, recall y clase correcta. Al final, une las grafías SEG que son la misma empresa que un mismo registro ORBIS.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  8 · REVISIÓN ADJUDICADA (opcional) Y MEDICIÓN DEL AUTOMÁTICO
# ══════════════════════════════════════════════════════════════════════════
revision_vigente()
FINAL, MEDICION = {}, []
for d in ("ORBIS", "FDI"):
    FINAL[d] = aplicar_revision_cruce(AUTO[d], REVISION, d)
    MEDICION.append(metricas_automatico(AUTO[d], REVISION, d))
for k in ("SEG", "ORBIS", "FDI"):
    m = metricas_dedup(PREP[k], REVISION, k, CLAVE[k])
    if m:
        MEDICION.append(m)
MEDICION = pd.DataFrame(MEDICION)
if REVISION is not None:
    print("📏 Motor automático contra la referencia adjudicada (sin la revisión aplicada):")
    print(MEDICION.to_string(index=False))
    for d in ("ORBIS", "FDI"):
        print(f"\n{d}: origen de los enlaces finales →", FINAL[d].ORIGEN.value_counts().to_dict())
else:
    print("Sin revisión: los enlaces finales son los automáticos. Revise la hoja REVISION_PENDIENTE del Excel.")

## 9 · Tablas e invariantes
Construye las hojas y verifica lo que siempre debe cumplirse. **Si una invariante falla, el notebook se detiene y el resultado no debe usarse.**

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  9 · TABLAS DE RESULTADO E INVARIANTES
# ══════════════════════════════════════════════════════════════════════════
revision_vigente()
# Dos grafías SEG que son la misma empresa que un mismo registro ORBIS (una persona jurídica) son la
# misma empresa. Se recalcula desde las uniones de la celda 6: correr esta celda dos veces da lo mismo.
extra = aristas_por_destino(FINAL["ORBIS"], CLAVE["SEG"])
PREP["SEG"], DEDUP_VETOS["SEG"] = agrupar(PREP["SEG"], pd.concat([ARISTAS["SEG"], extra], ignore_index=True), REVISION, "SEG", CLAVE["SEG"])
if len(extra):
    print(f"SEG: {len(extra)} uniones por ser la misma empresa que un mismo registro ORBIS → {PREP['SEG'].GRUPO.nunique()} entidades\n")
TABLAS, INVARIANTES = construir_tablas(CRUDO, PREP, FINAL, PARES, REVISION, MEDICION, FUENTES_SPEC, CFG, SIN_CONTRAPARTE,
                                       DEDUP_PEND, DEDUP_VETOS)
print(INVARIANTES.to_string(index=False))
if not INVARIANTES.cumple.all():
    raise RuntimeError("❌ Hay invariantes que no se cumplen: no use el resultado. Ver arriba.")
print("\n" + TABLAS["RESUMEN"].to_string(index=False))

## 10 · Exportación
Escribe el Excel con la ruta de exportación de la librería (neutraliza fórmulas y bytes de control) y la hoja de parámetros.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  10 · EXPORTACIÓN
# ══════════════════════════════════════════════════════════════════════════
sello = datetime.now().strftime("%Y%m%d_%H%M") if VERSIONAR_SALIDA else ""
nombre = f"match_segmentacion_orbis_fdi{('_' + sello) if sello else ''}"
TABLAS["PARAMETROS"] = parametros(CFG, FUENTES_SPEC, HUELLAS, ARCHIVO_REVISION, time.time() - T0)
RUTA_XLSX = escribir_excel(TABLAS, Path(CARPETA_SALIDA) / f"{nombre}.xlsx")
print(f"💾 {RUTA_XLSX}  ({RUTA_XLSX.stat().st_size / 2**20:.1f} MiB) · hojas: {', '.join(TABLAS)}")
print(f"⏱️  total {time.time() - T0:.1f} s")